In [2]:
%pip install lxml

  Using cached lxml-6.1.3-cp313-cp313-win_amd64.whl.metadata (3.4 kB)
Using cached lxml-6.1.3-cp313-cp313-win_amd64.whl (4.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd

df = pd.read_xml("../data/Entitats_ONGD_pretty.xml")
df.head()

Out[0]: 
   n_mero_registre  ...                          web
0            306.0  ...      http://ca180gradosinfo/
1            535.0  ...  http://www1x1microcreditorg
2            379.0  ...       http://wwweltriangleeu
3            318.0  ...                          NaN
4            459.0  ...           http://wwwigmancat

[5 rows x 11 columns]


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 765 entries, 0 to 764
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   n_mero_registre    765 non-null    float64
 1   nom_de_l_entitat   765 non-null    str    
 2   naturalesa         765 non-null    str    
 3   cif_entitat        682 non-null    str    
 4   adre_a             765 non-null    str    
 5   codi_postal        765 non-null    int64  
 6   municipi           765 non-null    str    
 7   tel_fon            544 non-null    str    
 8   tel_fon_m_bil      553 non-null    str    
 9   correu_electr_nic  730 non-null    str    
 10  web                611 non-null    str    
dtypes: float64(1), int64(1), str(9)
memory usage: 65.9 KB


## Exploratory analysis / Análisis exploratorio

Same workflow as `datos_abiertos.ipynb` and `entidades_CV.ipynb`: basic checks, field-specific checks, relationships, and conclusions after each column. Only observations are recorded; the XML and `df` are not cleaned, corrected, or transformed. Masks, lengths and counts are diagnostic results only.

Se sigue el mismo formato: comprobación básica, detalles de la columna, relaciones y conclusiones. No se modifica el XML ni `df`. Las comprobaciones de formato no acreditan validez fiscal, existencia de contactos ni exactitud geográfica. Los resultados se refieren exclusivamente a este archivo.

In [3]:
from pathlib import Path
import xml.etree.ElementTree as ET
from urllib.parse import urlsplit
from dataclasses import fields
import importlib.util

xml_path = Path("../data/Entitats_ONGD_pretty.xml")
xml_root = ET.parse(xml_path).getroot()
xml_rows = list(xml_root)
# Separate literal view of XML text; no assignments to df.
xml_text = pd.DataFrame([
    {element.tag: element.text for element in row}
    for row in xml_rows
])
df_snapshot = df.copy(deep=True)
print("XML root:", xml_root.tag)
print("XML records:", len(xml_rows))
print("Row tags:", {row.tag for row in xml_rows})
print("Same XML column order in all rows:",
      all([element.tag for element in row] == list(df.columns) for row in xml_rows))
print("Nested elements:", sum(bool(list(element)) for row in xml_rows for element in row))
print("XML attributes:", sum(len(element.attrib) for element in xml_root.iter()))
print("Rows / columns:", df.shape)
print("Exact duplicate rows:", df.duplicated().sum())
display(df.isna().sum())
display(df.nunique())
import re
# Display all compared fields; this changes presentation only.
pd.set_option("display.max_columns", None)


def repeated_value_summary(column):
    """Exact source strings only. Missing grouping keys are excluded."""
    counts = df[column].value_counts()
    keys = counts[counts.gt(1)].index
    rows = df.loc[df[column].notna() & df[column].isin(keys)]
    summary = rows.groupby(column).agg(
        rows=("n_mero_registre", "size"),
        registrations=("n_mero_registre", "nunique"),
        names=("nom_de_l_entitat", "nunique"),
        municipalities=("municipi", "nunique"),
        addresses=("adre_a", "nunique"),
        known_CIFs=("cif_entitat", "nunique"),
        missing_CIFs=("cif_entitat", lambda values: values.isna().sum()),
    )
    return rows, summary

def phone_category(value):
    # Mutually exclusive categories; priority follows this order.
    if pd.isna(value):
        return "missing"
    if re.fullmatch(r"[0-9]{9}", value):
        return "single nine digits"
    if "\n" in value:
        return "multiline"
    if re.search(r"(?i)ext|\([0-9]+\)", value):
        return "extension, single line"
    if "/" in value:
        return "slash, single line"
    if re.fullmatch(r"\+?34[0-9]{9}", value):
        return "country prefix, single line"
    return "other"

def email_category(value):
    # Classifies whole cells, not individual reconstructed contacts.
    if pd.isna(value):
        return "missing"
    at_count = value.count("@")
    if at_count == 0:
        return "no @"
    if at_count > 1:
        return "multiple @"
    if "." not in value.split("@", 1)[1]:
        return "single @, domain without dot"
    if re.fullmatch(r"[^\s@]+@[^\s@]+\.[^\s@]+", value):
        return "simple single-address pattern"
    return "single @, other format"


XML root: registre_ongd
XML records: 765
Row tags: {'entitat'}
Same XML column order in all rows: True
Nested elements: 0
XML attributes: 0
Rows / columns: (765, 11)
Exact duplicate rows: 0


n_mero_registre        0
nom_de_l_entitat       0
naturalesa             0
cif_entitat           83
adre_a                 0
codi_postal            0
municipi               0
tel_fon              221
tel_fon_m_bil        212
correu_electr_nic     35
web                  154
dtype: int64

n_mero_registre      765
nom_de_l_entitat     765
naturalesa            12
cif_entitat          680
adre_a               755
codi_postal          209
municipi             113
tel_fon              510
tel_fon_m_bil        543
correu_electr_nic    726
web                  607
dtype: int64

### Initial observations / Observaciones iniciales

- 765 `<entitat>` records, 11 columns; no exact duplicate rows, nested fields or XML attributes. The fields occur in the same order in every record.
- Empty XML elements occur in CIF, telephone, mobile, email and web fields; pandas represents these as missing values.
- Numeric inference affects registration numbers and postal codes. Literal XML text is inspected separately in `xml_text` without replacing the loaded data.

- Hay 765 registros y 11 columnas; no hay filas completamente duplicadas, campos anidados ni atributos XML.
- Los elementos XML vacíos se observan como valores ausentes en pandas. Se distingue el texto literal del XML del tipo inferido por pandas.

## `n_mero_registre`

In [4]:
col = 'n_mero_registre'
s = df[col]
print("dtype:", s.dtype)
print("missing:", s.isna().sum())
print("non-null:", s.notna().sum())
print("unique:", s.nunique())
display(s.dropna().head(10))
display(s.value_counts(dropna=False).head(20))

dtype: float64
missing: 0
non-null: 765
unique: 765


0    306.0
1    535.0
2    379.0
3    318.0
4    459.0
5    280.0
6    861.0
7     70.0
8     41.0
9    336.0
Name: n_mero_registre, dtype: float64

n_mero_registre
306.0    1
535.0    1
379.0    1
318.0    1
459.0    1
280.0    1
861.0    1
70.0     1
41.0     1
336.0    1
565.0    1
688.0    1
466.0    1
354.0    1
348.0    1
264.0    1
209.0    1
231.0    1
314.0    1
224.0    1
Name: count, dtype: int64

In [5]:
print("Unique pandas values:", s.is_unique)
print("Unique XML text:", xml_text[col].is_unique)
print("Fractional pandas values:", s.mod(1).ne(0).sum())
mask = s.mod(1).ne(0)
display(pd.DataFrame({"XML text": xml_text.loc[mask, col], "pandas value": s[mask],
                      "name": df.loc[mask, "nom_de_l_entitat"]}))
print("Literal XML formats:")
display(xml_text[col].str.len().value_counts())
print("Names per registration:")
display(df.groupby(col)["nom_de_l_entitat"].nunique().value_counts())
print("Registrations per name:")
display(df.groupby("nom_de_l_entitat")[col].nunique().value_counts())

Unique pandas values: True
Unique XML text: True
Fractional pandas values: 27


,XML text,pandas value,name
23,1.005,1.005,AsocaicionFinantiaEducatioSustine
32,1.002,1.002,AsociaciónBuenaventuraunMardeOportunidades
53,1.009,1.009,AsociacióndeMédicosPalestinosenEuropa-PALMEDES...
59,1.015,1.015,AsociacionDiásporadelaCommunedeKolibantang(ADCK)
88,1.018,1.018,AsociaciónPlataformaSocialTangarana
115,1.039,1.039,AssociacióAcciona't
211,1.031,1.031,AssociacióCulturalelParlante
220,1.011,1.011,"Associaciód'Ajudaal'Immigrant2009,Desenvolupam..."
228,1.016,1.016,AssociacióDavidSmile
283,1.045,1.045,AssociacióGermanetesJardinsd’Emma


Literal XML formats:


n_mero_registre
3    653
2     77
5     27
1      8
Name: count, dtype: int64

Names per registration:


nom_de_l_entitat
1    765
Name: count, dtype: int64

Registrations per name:


n_mero_registre
1    765
Name: count, dtype: int64

In [6]:

print("Literal XML registration formats:")
registration_formats = pd.DataFrame({
    "digits only": xml_text["n_mero_registre"].str.fullmatch(r"[0-9]+", na=False),
    "one dot, three trailing digits": xml_text["n_mero_registre"].str.fullmatch(r"[0-9]+\.[0-9]{3}", na=False),
})
display(registration_formats.sum().rename("rows"))
# Rows may differ only by identifiers: inspect that possibility separately.
non_id_columns = [column for column in df.columns
                  if column not in ["n_mero_registre", "nom_de_l_entitat", "cif_entitat"]]
same_non_id = df.duplicated(subset=non_id_columns, keep=False)
print("Rows with exactly equal non-identifier attributes:", same_non_id.sum())
display(df.loc[same_non_id])


Literal XML registration formats:


digits only                       738
one dot, three trailing digits     27
Name: rows, dtype: int64

Rows with exactly equal non-identifier attributes: 0


,n_mero_registre,nom_de_l_entitat,naturalesa,cif_entitat,adre_a,codi_postal,municipi,tel_fon,tel_fon_m_bil,correu_electr_nic,web


### `n_mero_registre` — conclusion

- 765 non-null values; all are unique in pandas and as literal XML text. Registration and name are one-to-one in this file.
- pandas inferred `float64`: 738 literal values contain digits only; 27 contain a dot followed by three digits, e.g. `1.005`.
- The dot may be a thousands separator; its meaning is unconfirmed. No arithmetic interpretation or correction is applied.
- No two rows have identical values across all non-identifier fields after excluding registration, name and CIF.
- Candidate for `Entidad.cod_entidad` (`str`). Uniqueness across sources and the final identifier representation remain unresolved.

### `n_mero_registre` — conclusión

- 765 valores no nulos; todos únicos en pandas y en el XML literal. Registro y nombre tienen relación uno a uno en este archivo.
- pandas infiere `float64`: 738 textos contienen solo dígitos y 27 un punto seguido de tres dígitos, por ejemplo `1.005`.
- El punto podría separar miles; su significado no está confirmado. No se interpreta como magnitud ni se corrige.
- Al excluir registro, nombre y CIF, no hay dos filas iguales en todos los demás campos.
- Candidato a `Entidad.cod_entidad` (`str`). Quedan abiertas la unicidad entre fuentes y la representación final.

## `nom_de_l_entitat`

In [7]:
col = 'nom_de_l_entitat'
s = df[col]
print("dtype:", s.dtype)
print("missing:", s.isna().sum())
print("non-null:", s.notna().sum())
print("unique:", s.nunique())
display(s.dropna().head(10))
display(s.value_counts(dropna=False).head(20))
print("Blank values:", s.str.fullmatch(r"\s*", na=False).sum())
print("Leading/trailing whitespace:", (s.notna() & s.ne(s.str.strip())).sum())
print("Whitespace anywhere:", s.str.contains(r"\s", na=False).sum())
print("Literal placeholder candidates (not recoded):")
display(s[s.isin(["_U", "NULL", "Null", "null", "None", "N/A", "-", "0"])].value_counts())

dtype: str
missing: 0
non-null: 765
unique: 765


0                   180ºParaLaCooperaciónYElDesarrollo
1                                       1X1Microcredit
2                                          AccióPremsa
3                              AccióSolidàriadelVallès
4                                  AccióSolidàriaIgman
5              ACOES-AsociacióndeColaboraciónyEsfuerzo
6                                   AfrowestAsociacion
7                                     AgermanamentPerú
8                           AgermanamentsenseFronteres
9    ÀgoraNord-Sud(OngCatalanesDeSolidaritatInterna...
Name: nom_de_l_entitat, dtype: str

nom_de_l_entitat
180ºParaLaCooperaciónYElDesarrollo                       1
1X1Microcredit                                           1
AccióPremsa                                              1
AccióSolidàriadelVallès                                  1
AccióSolidàriaIgman                                      1
ACOES-AsociacióndeColaboraciónyEsfuerzo                  1
AfrowestAsociacion                                       1
AgermanamentPerú                                         1
AgermanamentsenseFronteres                               1
ÀgoraNord-Sud(OngCatalanesDeSolidaritatInternacional)    1
AjutalPobleCubàaLaHabana                                 1
AlegríaSinFronteras(ASF)                                 1
AlianzadeSolidaridadExtremeñaASE                         1
AllianceforFreedomandDignity,AFDEspaña                   1
AMAMESPAÑA-AsociacióndeMujeresAnti-mutilaciones          1
AMAZAN:Promociósocioculturalidesenvolupament             1
AMAZÒNIACAT                            

Blank values: 0
Leading/trailing whitespace: 0
Whitespace anywhere: 1
Literal placeholder candidates (not recoded):


Series([], Name: count, dtype: int64)

In [8]:
length_profile = pd.DataFrame({col: s, "length": s.str.len()})
display(length_profile["length"].describe())
with pd.option_context("display.max_colwidth", None):
    display(length_profile.nsmallest(10, "length"))
    display(length_profile.nlargest(10, "length"))
print("Differences from literal XML:",
      (s.notna() & s.ne(xml_text[col])).sum())

count    765.000000
mean      39.577778
std       21.167143
min        5.000000
25%       24.000000
50%       35.000000
75%       51.000000
max      165.000000
Name: length, dtype: float64

,nom_de_l_entitat,length
497,CESAL,5
737,PIMEC,5
534,EDUGASCAR,9
750,RifTribes,9
499,CivesMundi,10
706,GrupEirene,10
747,RedActivas,10
2,AccióPremsa,11
16,AMAZÒNIACAT,11
592,FundacióMón,11


,nom_de_l_entitat,length
572,"FundacióCaminsFundacióSocialEscolaPia(Absorcióde:FundacióPrivadaBenallar,FundacióPrivadaEducacióSolidària-EscolaPia,FundacióPrivadaServeiSolidariperlaInclusióSocial)",165
612,"FundaciónFinanzasÉticasActivismoyEconomíaCrítica–FundacióFinancesÈtiquesActivismeiEconomiaCrítica–FinantzaEtikoak,FundazioaAktibismoetaEkonomiaKritikoa",151
363,"AssociacióNAGA-CAW(NetworkforAfricanGrowth,AdvancementandCommunityAssistanceWelfare/Redparaelcrecimiento,eldesarrolloyelbienestarsocialenÁfrica)",144
544,FederacióCoordinadorad’ONGSolidàriesdelescomarquesgironinesil’AltMaresme(Antiga:AssociacióCentredeRecursosperalaPauilaSolidaritat(CeDRe),136
543,FederacióaFFaC–AssociacionsFederadesdeFamíliesd’AlumnesdeCatalunya(Antiga:Federaciód'AssociacionsdeMaresiParesd'AlumnesdeCatalunya),131
380,"AssociacióObservatorisobreelcomplimentdelaDeclaracióUniversaldelsDretsHumansenelsDretsEconòmics,SocialsiCulturals(ObservatoriDESC)",130
731,"NexesInterculturals,Sccl(Abans:NexesInterculturalsDeJovesPerEuropa-ProgramesDeFormació,CulturaIDesenvolupamentEconòmic)",119
177,"AssociacióCATESCO(CataloniaforEducation,ScienceandCultureOrganization)-(AntigaAssociacióCentreUNESCOdeCatalunya)",112
546,FederacióCoordinadorad'OrganitzacionsNoGovernamentalsperalDesenvolupament(ONGD)ialtresMovimentsSolidarisdeLleida,112
542,"Fassulò,AsociaciónSocioculturalSenegalesaporlaLuchacontralaPobresayporelDesarrollodelosPueblosDesfavorecidos",108


Differences from literal XML: 0


In [9]:
display(df.loc[s.str.contains(r"\s", na=False), ["n_mero_registre", col]])
print("Registrations per name:")
display(df.groupby(col)["n_mero_registre"].nunique().value_counts())

,n_mero_registre,nom_de_l_entitat
687,254.0,FundacióPrivadaUtopia(AbsorveixL'AntigaAssocia...


Registrations per name:


n_mero_registre
1    765
Name: count, dtype: int64

In [10]:

print("Literal representation of the name containing whitespace:")
name_whitespace = df["nom_de_l_entitat"].str.contains(r"\s", na=False)
display(df.loc[name_whitespace, ["n_mero_registre", "nom_de_l_entitat"]]
        .assign(literal_repr=df.loc[name_whitespace, "nom_de_l_entitat"].map(repr)))
print("Longest name endings (inspection only, no truncation inferred):")
longest_names = df["nom_de_l_entitat"].str.len().nlargest(5).index
display(pd.DataFrame({
    "name": df.loc[longest_names, "nom_de_l_entitat"],
    "length": df.loc[longest_names, "nom_de_l_entitat"].str.len(),
    "last_50_characters": df.loc[longest_names, "nom_de_l_entitat"].str[-50:],
}))


Literal representation of the name containing whitespace:


,n_mero_registre,nom_de_l_entitat,literal_repr
687,254.0,FundacióPrivadaUtopia(AbsorveixL'AntigaAssocia...,"""FundacióPrivadaUtopia(AbsorveixL'AntigaAssoci..."


Longest name endings (inspection only, no truncation inferred):


,name,length,last_50_characters
572,FundacióCaminsFundacióSocialEscolaPia(Absorció...,165,",FundacióPrivadaServeiSolidariperlaInclusióSoc..."
612,FundaciónFinanzasÉticasActivismoyEconomíaCríti...,151,"antzaEtikoak,FundazioaAktibismoetaEkonomiaKrit..."
363,"AssociacióNAGA-CAW(NetworkforAfricanGrowth,Adv...",144,"recimiento,eldesarrolloyelbienestarsocialenÁfr..."
544,FederacióCoordinadorad’ONGSolidàriesdelescomar...,136,acióCentredeRecursosperalaPauilaSolidaritat(Ce...
543,FederacióaFFaC–AssociacionsFederadesdeFamílies...,131,iód'AssociacionsdeMaresiParesd'AlumnesdeCatalu...


### `nom_de_l_entitat` — conclusion

- 765 populated, distinct names; lengths 5–165. No leading/trailing whitespace.
- 764 names contain no whitespace; the remaining name contains a newline. All names match literal XML text.
- Compound names join words, e.g. `AccióSolidàriadelVallès`; this can hinder reading and matching. Single-word names need no spaces.
- Length extremes and endings are displayed for inspection; these checks alone do not establish truncation.
- Candidate for `Entidad.nombre`. Observed name uniqueness does not establish a universal entity identifier.

### `nom_de_l_entitat` — conclusión

- 765 nombres informados y distintos; longitudes de 5 a 165. Sin espacios iniciales/finales.
- 764 nombres no contienen caracteres de espacio; el restante contiene un salto de línea. Todos coinciden con el XML literal.
- Hay palabras concatenadas, como `AccióSolidàriadelVallès`, que pueden dificultar lectura y comparación. Los nombres de una palabra no requieren espacios.
- Se muestran longitudes extremas y finales para inspección; estas comprobaciones no demuestran truncamiento.
- Candidato a `Entidad.nombre`; su unicidad aquí no demuestra identidad universal.

## `naturalesa`

In [11]:
col = 'naturalesa'
s = df[col]
print("dtype:", s.dtype)
print("missing:", s.isna().sum())
print("non-null:", s.notna().sum())
print("unique:", s.nunique())
display(s.dropna().head(10))
display(s.value_counts(dropna=False).head(20))
print("Blank values:", s.str.fullmatch(r"\s*", na=False).sum())
print("Leading/trailing whitespace:", (s.notna() & s.ne(s.str.strip())).sum())
print("Whitespace anywhere:", s.str.contains(r"\s", na=False).sum())
print("Literal placeholder candidates (not recoded):")
display(s[s.isin(["_U", "NULL", "Null", "null", "None", "N/A", "-", "0"])].value_counts())

dtype: str
missing: 0
non-null: 765
unique: 12


0    Associació
1    Associació
2    Associació
3    Associació
4    Associació
5    Associació
6    Associació
7    Associació
8    Associació
9    Associació
Name: naturalesa, dtype: str

naturalesa
Associació                    536
Fundació                      132
AssociacióEstatal              45
Cooperativa                    22
Federació                      12
FundacióEstatal                10
DelegacióFundacióNacional       3
DelegacióAssociació             1
DelegacióAssociacióEstatal      1
DelegacióAutonòmica             1
EntitatReligiosa                1
DelegacióFundació               1
Name: count, dtype: int64

Blank values: 0
Leading/trailing whitespace: 0
Whitespace anywhere: 0
Literal placeholder candidates (not recoded):


Series([], Name: count, dtype: int64)

In [12]:
display(s.value_counts(dropna=False))
display(pd.crosstab(df[col], df["cif_entitat"].isna(),
                    colnames=["CIF missing"]))

naturalesa
Associació                    536
Fundació                      132
AssociacióEstatal              45
Cooperativa                    22
Federació                      12
FundacióEstatal                10
DelegacióFundacióNacional       3
DelegacióAssociació             1
DelegacióAssociacióEstatal      1
DelegacióAutonòmica             1
EntitatReligiosa                1
DelegacióFundació               1
Name: count, dtype: int64

CIF missing,False,True
naturalesa,,
Associació,463,73
AssociacióEstatal,44,1
Cooperativa,22,0
DelegacióAssociació,1,0
DelegacióAssociacióEstatal,1,0
DelegacióAutonòmica,1,0
DelegacióFundació,1,0
DelegacióFundacióNacional,3,0
EntitatReligiosa,1,0


In [13]:

availability_columns = ["cif_entitat", "tel_fon", "tel_fon_m_bil", "correu_electr_nic", "web"]
nature_sizes = df.groupby("naturalesa").size().rename("rows")
missing_by_nature = df[availability_columns].isna().groupby(df["naturalesa"]).sum()
missing_pct_by_nature = missing_by_nature.div(nature_sizes, axis=0).mul(100).round(1)
print("Missing-value counts by legal form (group size included):")
display(nature_sizes.to_frame().join(missing_by_nature))
print("Missing-value percentages within each legal-form group:")
display(nature_sizes.to_frame().join(missing_pct_by_nature.add_suffix(" missing %")))


Missing-value counts by legal form (group size included):


,rows,cif_entitat,tel_fon,tel_fon_m_bil,correu_electr_nic,web
naturalesa,,,,,,
Associació,536,73,176,127,33,135
AssociacióEstatal,45,1,14,16,1,4
Cooperativa,22,0,14,3,0,0
DelegacióAssociació,1,0,0,0,0,0
DelegacióAssociacióEstatal,1,0,0,1,0,0
DelegacióAutonòmica,1,0,0,1,0,0
DelegacióFundació,1,0,0,0,0,0
DelegacióFundacióNacional,3,0,0,1,0,0
EntitatReligiosa,1,0,0,1,0,0


Missing-value percentages within each legal-form group:


,rows,cif_entitat missing %,tel_fon missing %,tel_fon_m_bil missing %,correu_electr_nic missing %,web missing %
naturalesa,,,,,,
Associació,536,13.6,32.8,23.7,6.2,25.2
AssociacióEstatal,45,2.2,31.1,35.6,2.2,8.9
Cooperativa,22,0.0,63.6,13.6,0.0,0.0
DelegacióAssociació,1,0.0,0.0,0.0,0.0,0.0
DelegacióAssociacióEstatal,1,0.0,0.0,100.0,0.0,0.0
DelegacióAutonòmica,1,0.0,0.0,100.0,0.0,0.0
DelegacióFundació,1,0.0,0.0,0.0,0.0,0.0
DelegacióFundacióNacional,3,0.0,0.0,33.3,0.0,0.0
EntitatReligiosa,1,0.0,0.0,100.0,0.0,0.0


### `naturalesa` — conclusion

- Complete categorical field: 12 values; `Associació` 536 rows, `Fundació` 132.
- Compound labels lack separating spaces, e.g. `AssociacióEstatal`. Single-word labels are not anomalous for lacking spaces.
- Missingness differs descriptively by legal form: CIF is missing in 73/536 associations (13.6%) and 9/132 foundations (6.8%). Mobile is missing in 127/536 (23.7%) and 58/132 (43.9%), respectively.
- Group sizes and rates are displayed; small categories do not support general conclusions or causal claims.
- This is legal/organisational nature, not an `Ambito`. The schema has no dedicated legal-form attribute.

### `naturalesa` — conclusión

- Campo categórico completo: 12 valores; `Associació` aparece en 536 filas y `Fundació` en 132.
- Etiquetas compuestas sin espacios, como `AssociacióEstatal`; no es anomalía en etiquetas de una palabra.
- Ausencias por naturaleza: CIF en 73/536 asociaciones (13,6%) y 9/132 fundaciones (6,8%); móvil en 127/536 (23,7%) y 58/132 (43,9%), respectivamente.
- Se muestran tamaños y porcentajes por grupo. Las categorías pequeñas no permiten generalizaciones ni conclusiones causales.
- Describe naturaleza jurídica/organizativa, no `Ambito`. No hay atributo específico en el esquema.

## `cif_entitat`

In [14]:
col = 'cif_entitat'
s = df[col]
print("dtype:", s.dtype)
print("missing:", s.isna().sum())
print("non-null:", s.notna().sum())
print("unique:", s.nunique())
display(s.dropna().head(10))
display(s.value_counts(dropna=False).head(20))
print("Blank values:", s.str.fullmatch(r"\s*", na=False).sum())
print("Leading/trailing whitespace:", (s.notna() & s.ne(s.str.strip())).sum())
print("Whitespace anywhere:", s.str.contains(r"\s", na=False).sum())
print("Literal placeholder candidates (not recoded):")
display(s[s.isin(["_U", "NULL", "Null", "null", "None", "N/A", "-", "0"])].value_counts())

dtype: str
missing: 83
non-null: 682
unique: 680


0     G63749444
3     G65049017
4     G62117494
5     G62730114
6     G44928257
7     G17646548
8     G61221859
9     G63174486
10    G61178885
11    G66107434
Name: cif_entitat, dtype: str

cif_entitat
NaN          83
G10932812     2
G55258909     2
G63749444     1
G65049017     1
G62117494     1
G62730114     1
G44928257     1
G17646548     1
G61221859     1
G63174486     1
G61178885     1
G66107434     1
G10184653     1
G05207386     1
G62542709     1
G64132293     1
G63574123     1
G64176134     1
G61898755     1
Name: count, dtype: int64

Blank values: 0
Leading/trailing whitespace: 0
Whitespace anywhere: 0
Literal placeholder candidates (not recoded):


Series([], Name: count, dtype: int64)

In [15]:
display(s.str.len().value_counts(dropna=False))
pattern = r"[A-Z][0-9]{7}[0-9A-Z]"
format_exception = s.notna() & ~s.str.fullmatch(pattern, na=False)
display(df.loc[format_exception, ["n_mero_registre", "nom_de_l_entitat", col]])
repeated_cif = s.notna() & s.duplicated(keep=False)
display(df.loc[repeated_cif,
               [col, "n_mero_registre", "nom_de_l_entitat", "adre_a", "municipi"]].sort_values(col))
print("Registrations, names and municipalities per populated CIF:")
display(df.loc[s.notna()].groupby(col)[
    ["n_mero_registre", "nom_de_l_entitat", "municipi"]].nunique().value_counts())

cif_entitat
9.0    682
NaN     83
Name: count, dtype: int64

,n_mero_registre,nom_de_l_entitat,cif_entitat
423,999.0,AssociacióRedGirlsProject,46414697F


,cif_entitat,n_mero_registre,nom_de_l_entitat,adre_a,municipi
100,G10932812,832.0,AsociaciónSolidariaInternacionalporlaLuchadeOp...,"CarrerTarragona,20",Teià
364,G10932812,833.0,AssociacióÑamatu(MareTerra),"PlaçaAssumpció,27",Girona
136,G55258909,838.0,AssociacióAmbLesTevesMans,"CarrerRectoria,6",Armentera(L')
552,G55258909,720.0,FederacióndelasAsociacionesdeAfricaNegradeSaba...,"CarrerSantJoan,20baixos",Sabadell


Registrations, names and municipalities per populated CIF:


n_mero_registre  nom_de_l_entitat  municipi
1                1                 1           678
2                2                 2             2
Name: count, dtype: int64

In [16]:

cif_rows, cif_summary = repeated_value_summary("cif_entitat")
display(cif_summary)
print("Full records for each repeated CIF:")
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(cif_rows.sort_values(["cif_entitat", "n_mero_registre"]))
print("Distinct values per field, counting missing as a value:")
cif_variation = cif_rows.groupby("cif_entitat").nunique(dropna=False)
display(cif_variation)
print("Fields that vary in each repeated-CIF group:")
display(pd.Series({key: list(values.index[values.gt(1)])
                   for key, values in cif_variation.iterrows()}, name="varying fields"))


,rows,registrations,names,municipalities,addresses,known_CIFs,missing_CIFs
cif_entitat,,,,,,,
G10932812,2,2,2,2,2,1,0
G55258909,2,2,2,2,2,1,0


Full records for each repeated CIF:


,n_mero_registre,nom_de_l_entitat,naturalesa,cif_entitat,adre_a,codi_postal,municipi,tel_fon,tel_fon_m_bil,correu_electr_nic,web
100,832.0,AsociaciónSolidariaInternacionalporlaLuchadeOportunidadesenGuineaBissau,Associació,G10932812,"CarrerTarragona,20",8329,Teià,NaN,NaN,rosagsaez2407@gmailcom,NaN
364,833.0,AssociacióÑamatu(MareTerra),Associació,G10932812,"PlaçaAssumpció,27",17005,Girona,NaN,689983511,associacionamatu@gmailcom,NaN
552,720.0,FederacióndelasAsociacionesdeAfricaNegradeSabadellyCatalunya(FEDEAFRICANESCA),Federació,G55258909,"CarrerSantJoan,20baixos",8202,Sabadell,664516313,632291959,fedeafricanesca@gmailcom,NaN
136,838.0,AssociacióAmbLesTevesMans,Associació,G55258909,"CarrerRectoria,6",17472,Armentera(L'),NaN,661512034,gestio@amblestevesmanscat,NaN


Distinct values per field, counting missing as a value:


,n_mero_registre,nom_de_l_entitat,naturalesa,adre_a,codi_postal,municipi,tel_fon,tel_fon_m_bil,correu_electr_nic,web
cif_entitat,,,,,,,,,,
G10932812,2,2,1,2,2,2,1,2,2,1
G55258909,2,2,2,2,2,2,2,2,2,1


Fields that vary in each repeated-CIF group:


G10932812    [n_mero_registre, nom_de_l_entitat, adre_a, co...
G55258909    [n_mero_registre, nom_de_l_entitat, naturalesa...
Name: varying fields, dtype: object

### `cif_entitat` — conclusion

- 682 populated values, 83 missing (10.8%); 680 distinct populated CIFs. All populated strings have nine characters.
- 681 match the illustrative organisation-ID pattern. `46414697F` has a DNI-like shape; neither checksums nor legal identity are validated.
- Two CIFs repeat, covering four rows. Both groups differ in registration, name, address, postal code, municipality, mobile and email.
- `G55258909` also differs in legal form (`Associació` / `Federació`) and telephone availability; `G10932812` has `Associació` in both rows and no telephone in either. Web is missing in all four rows.
- The full-field comparison does not establish whether these are related organisations or source errors. Shared CIF alone does not justify merging.
- Incomplete and non-unique as a row key; no dedicated CIF/NIF attribute exists in the schema.

### `cif_entitat` — conclusión

- 682 valores informados, 83 ausentes (10,8%) y 680 CIF distintos. Todos los textos informados tienen nueve caracteres.
- 681 cumplen el patrón orientativo de organización. `46414697F` tiene forma similar a DNI; no se valida control ni identidad jurídica.
- Dos CIF repetidos afectan a cuatro filas. Ambos grupos difieren en registro, nombre, dirección, código postal, municipio, móvil y correo.
- `G55258909` también difiere en naturaleza (`Associació` / `Federació`) y disponibilidad de teléfono. `G10932812` tiene `Associació` en ambas filas y carece de teléfono en las dos. Las cuatro carecen de web.
- La comparación completa no determina si son organizaciones relacionadas o errores de fuente. Compartir CIF no basta para fusionar.
- No es clave completa y única; no hay atributo CIF/NIF específico en el esquema.

## `adre_a`

In [17]:
col = 'adre_a'
s = df[col]
print("dtype:", s.dtype)
print("missing:", s.isna().sum())
print("non-null:", s.notna().sum())
print("unique:", s.nunique())
display(s.dropna().head(10))
display(s.value_counts(dropna=False).head(20))
print("Blank values:", s.str.fullmatch(r"\s*", na=False).sum())
print("Leading/trailing whitespace:", (s.notna() & s.ne(s.str.strip())).sum())
print("Whitespace anywhere:", s.str.contains(r"\s", na=False).sum())
print("Literal placeholder candidates (not recoded):")
display(s[s.isin(["_U", "NULL", "Null", "null", "None", "N/A", "-", "0"])].value_counts())

dtype: str
missing: 0
non-null: 765
unique: 755


0        PladelVinyet,9BlocDbaixos
1         MajordeSarrià,125bis,3-3
2    PgFerrocarrilsCatalans,97-117
3                       Mallorca,2
4                      Fonollar,14
5        PasseigGarciaiFaria,335èB
6             CarrerPompeuFabra,14
7                         Llarg,56
8              LaRambla,10entresòl
9                       Pelai,522n
Name: adre_a, dtype: str

adre_a
Fonollar,14                      2
JosepAnselmClavé,61er1a          2
CarrerRipoll,25                  2
Acadèmia,44altell                2
CarrerNou,12                     2
Casp,43baixos                    2
PlaçaLluísCompanys,12            2
Sorrall,9                        2
CarrerParis,150,1r2a             2
LuisAntúnez,6                    2
PladelVinyet,9BlocDbaixos        1
MajordeSarrià,125bis,3-3         1
PgFerrocarrilsCatalans,97-117    1
Mallorca,2                       1
PasseigGarciaiFaria,335èB        1
CarrerPompeuFabra,14             1
Llarg,56                         1
LaRambla,10entresòl              1
Pelai,522n                       1
PlaçaCatalunya,51er1a            1
Name: count, dtype: int64

Blank values: 0
Leading/trailing whitespace: 0
Whitespace anywhere: 0
Literal placeholder candidates (not recoded):


Series([], Name: count, dtype: int64)

In [18]:
length_profile = pd.DataFrame({col: s, "length": s.str.len()})
display(length_profile["length"].describe())
with pd.option_context("display.max_colwidth", None):
    display(length_profile.nsmallest(10, "length"))
    display(length_profile.nlargest(10, "length"))
print("Differences from literal XML:",
      (s.notna() & s.ne(xml_text[col])).sum())

count    765.000000
mean      21.186928
std        7.280134
min        7.000000
25%       16.000000
50%       20.000000
75%       25.000000
max       66.000000
Name: length, dtype: float64

,adre_a,length
288,"Colom,2",7
7,"Llarg,56",8
290,"Firal,22",8
78,AvEgara35,9
291,"Sorrall,9",9
319,"LaGleva,9",9
327,Bagesnº11,9
337,"LaPalma,6",9
432,"Sorrall,9",9
583,"Canigó,33",9


,adre_a,length
464,"ViaAugusta13-15,3ªplantaoficina313(entradaporComtedeSalvatierra,5)",66
643,EdificiMRAParcdeRecercaUniversitatAutònomadeBarcelona(Bellaterra),65
569,PlaçaCívicaCampusdelaUniversitatAutònomadeBarcelona(UAB),56
657,"PasseigSalvatPapasseit,11a(EdificiLaFàbricadelSol)",50
165,"CentroCulturalSantJosep-AvIsabellaCatólica,32BJS",48
224,"PlaçaLesMissions,4(ParroquiaSanturaiStaTeresina)",48
49,"MossènAndreu,13-19CentreCulturalJoanGarciaNieto",47
260,"PlaçaEusebiGüell,6baixosEdifVèrtex,SalaPangea",45
358,"CarrerdePereVergès,1EdificiPiramidó-Local9-10",45
389,"PlaçaMargaridaXirgu,1(EdificioTeatreLluire)",43


Differences from literal XML: 0


In [19]:
display(df.loc[s.duplicated(keep=False),
               ["n_mero_registre", "nom_de_l_entitat", col, "codi_postal", "municipi"]].sort_values(col))
print("Municipalities per address:")
display(df.groupby(col)["municipi"].nunique().value_counts())

,n_mero_registre,nom_de_l_entitat,adre_a,codi_postal,municipi
172,226.0,AssociacióCatalanaperalDesenvolupamentilaCoope...,"Acadèmia,44altell",25002,Lleida
287,225.0,AssociacióGrupCulturalGarrigues,"Acadèmia,44altell",25002,Lleida
213,768.0,AssociacióCulturall'Aleta,"CarrerNou,12",43800,Valls
343,592.0,"AssociacióLlumull,AccionsAudiovisuals","CarrerNou,12",43800,Valls
747,493.0,RedActivas,"CarrerParis,150,1r2a",8036,Barcelona
493,452.0,Centred'AnàlisiiProgramesSanitaris(CAPS),"CarrerParis,150,1r2a",8036,Barcelona
76,917.0,AsociaciónMujerDiáspora,"CarrerRipoll,25",8002,Barcelona
93,617.0,"AsociaciónReddeMigración,GéneroyDesarrollo","CarrerRipoll,25",8002,Barcelona
279,197.0,AssociacióFETS-FinançamentÈticiSolidàri,"Casp,43baixos",8010,Barcelona
320,154.0,AssociacióJuristesSolidaris,"Casp,43baixos",8010,Barcelona


Municipalities per address:


municipi
1    755
Name: count, dtype: int64

In [20]:

address_rows, address_summary = repeated_value_summary("adre_a")
print("Repeated literal addresses: group counts and identifier context:")
display(address_summary)
print("Distinct values of ALL other fields within each address group (missing included):")
display(address_rows.groupby("adre_a").nunique(dropna=False))
print("All records in repeated-address groups:")
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(address_rows.sort_values(["adre_a", "n_mero_registre"]))


Repeated literal addresses: group counts and identifier context:


,rows,registrations,names,municipalities,addresses,known_CIFs,missing_CIFs
adre_a,,,,,,,
"Acadèmia,44altell",2,2,2,1,1,2,0
"CarrerNou,12",2,2,2,1,1,2,0
"CarrerParis,150,1r2a",2,2,2,1,1,1,1
"CarrerRipoll,25",2,2,2,1,1,2,0
"Casp,43baixos",2,2,2,1,1,2,0
"Fonollar,14",2,2,2,1,1,2,0
"JosepAnselmClavé,61er1a",2,2,2,1,1,1,1
"LuisAntúnez,6",2,2,2,1,1,1,1
"PlaçaLluísCompanys,12",2,2,2,1,1,2,0


Distinct values of ALL other fields within each address group (missing included):


,n_mero_registre,nom_de_l_entitat,naturalesa,cif_entitat,codi_postal,municipi,tel_fon,tel_fon_m_bil,correu_electr_nic,web
adre_a,,,,,,,,,,
"Acadèmia,44altell",2,2,1,2,1,1,1,2,2,1
"CarrerNou,12",2,2,1,2,1,1,2,2,2,2
"CarrerParis,150,1r2a",2,2,1,2,1,1,2,1,2,2
"CarrerRipoll,25",2,2,1,2,1,1,2,2,2,2
"Casp,43baixos",2,2,1,2,1,1,2,2,2,2
"Fonollar,14",2,2,2,2,1,1,1,1,2,2
"JosepAnselmClavé,61er1a",2,2,1,2,1,1,2,1,2,2
"LuisAntúnez,6",2,2,1,2,1,1,2,2,2,2
"PlaçaLluísCompanys,12",2,2,2,2,1,1,2,1,2,2


All records in repeated-address groups:


,n_mero_registre,nom_de_l_entitat,naturalesa,cif_entitat,adre_a,codi_postal,municipi,tel_fon,tel_fon_m_bil,correu_electr_nic,web
287,225.0,AssociacióGrupCulturalGarrigues,Associació,G25208265,"Acadèmia,44altell",25002,Lleida,973275415,NaN,garriguescooperacio@gmailcom,NaN
172,226.0,"AssociacióCatalanaperalDesenvolupamentilaCooperació,Lleidacat",Associació,G25642695,"Acadèmia,44altell",25002,Lleida,973275415,669632821,guiuestopa@hotmailcom,NaN
343,592.0,"AssociacióLlumull,AccionsAudiovisuals",Associació,G43967157,"CarrerNou,12",43800,Valls,977602456,620673774,llumullaccions@gmailcom,http://llumullblogspotcom
213,768.0,AssociacióCulturall'Aleta,Associació,G43519966,"CarrerNou,12",43800,Valls,NaN,666922556,info@laturbacat,http://wwwlaturbacat
493,452.0,Centred'AnàlisiiProgramesSanitaris(CAPS),Associació,G08830127,"CarrerParis,150,1r2a",8036,Barcelona,933226554,NaN,caps@pangeaorg,http://wwwcapscat
747,493.0,RedActivas,Associació,NaN,"CarrerParis,150,1r2a",8036,Barcelona,934228074\n93431426,NaN,catalunya@redactivasorgadministracion@redactivasorgxarxaactives@redactivesorgdirectora@redactivasorg,http://wwwredactivasorg
93,617.0,"AsociaciónReddeMigración,GéneroyDesarrollo",Associació,G66306226,"CarrerRipoll,25",8002,Barcelona,NaN,657430586,xarxamgd@gmailcom,http://wwwredmgdorg/
76,917.0,AsociaciónMujerDiáspora,Associació,G67375915,"CarrerRipoll,25",8002,Barcelona,699528604,646644983,mujerdiaspora@gmailcom,http://https://wwwmujerdiasporacom·
320,154.0,AssociacióJuristesSolidaris,Associació,G64436603,"Casp,43baixos",8010,Barcelona,935285990,605516286,juristesolidaris@gmailcom,NaN
279,197.0,AssociacióFETS-FinançamentÈticiSolidàri,Associació,G62069596,"Casp,43baixos",8010,Barcelona,933688481,NaN,fets@fetsorg,http://wwwfetsorg


### `adre_a` — conclusion

- 765 populated addresses, 755 distinct; lengths 7–66. None contains whitespace.
- Words and components appear joined, e.g. `PladelVinyet,9BlocDbaixos`; punctuation and component separation vary. This is literal XML text.
- 10 addresses repeat twice (20 rows). Each pair shares its municipality and postal code but has different registrations and names.
- Seven pairs contain two distinct known CIFs; in three pairs one CIF is missing. All remaining fields are compared in the displayed tables.
- Shared address does not establish duplicate entities. Candidate for `Entidad.direccion`; no components are parsed into a corrected address.

### `adre_a` — conclusión

- 765 direcciones informadas y 755 distintas; longitudes de 7 a 66. Ninguna contiene caracteres de espacio.
- Palabras y componentes unidos, como `PladelVinyet,9BlocDbaixos`; puntuación y separación variables, ya en el XML.
- 10 direcciones aparecen dos veces (20 filas). Cada pareja comparte municipio y código postal, pero tiene registros y nombres distintos.
- Siete parejas tienen dos CIF conocidos distintos; en tres falta uno de los CIF. Las tablas comparan los demás campos.
- Compartir dirección no demuestra duplicación. Candidato a `Entidad.direccion`; no se reconstruyen componentes.

## `codi_postal`

In [21]:
col = 'codi_postal'
s = df[col]
print("dtype:", s.dtype)
print("missing:", s.isna().sum())
print("non-null:", s.notna().sum())
print("unique:", s.nunique())
display(s.dropna().head(10))
display(s.value_counts(dropna=False).head(20))

dtype: int64
missing: 0
non-null: 765
unique: 209


0     8172
1     8017
2     8940
3     8192
4     8003
5     8005
6    17411
7    17196
8     8002
9     8001
Name: codi_postal, dtype: int64

codi_postal
8001     52
8003     30
8002     21
8006     21
8010     19
8029     18
8024     16
8012     16
8005     14
8014     14
8015     13
8025     12
8172     11
8034     11
8028     10
17005    10
8037     10
8017      9
8007      9
8020      9
Name: count, dtype: int64

In [22]:
literal_postal = xml_text[col]
display(literal_postal.str.len().value_counts())
print("Five digits in literal XML:", literal_postal.str.fullmatch(r"[0-9]{5}", na=False).sum())
print("XML values starting with zero:", literal_postal.str.startswith("0", na=False).sum())
print("pandas numeric range:", s.min(), s.max())
display(pd.DataFrame({"XML text": literal_postal, "pandas value": s}).head(15))
municipalities_per_postal = df.groupby(col)["municipi"].nunique()
print("Municipalities per postal code:")
display(municipalities_per_postal.value_counts())
multiple_postal = municipalities_per_postal[municipalities_per_postal.gt(1)].index
display(df.loc[s.isin(multiple_postal), [col, "municipi"]].value_counts().sort_index())
print("XML prefix frequencies (no province inference):")
display(literal_postal.str.slice(0, 2).value_counts())

codi_postal
5    765
Name: count, dtype: int64

Five digits in literal XML: 765
XML values starting with zero: 618
pandas numeric range: 800 43870


,XML text,pandas value
0,08172,8172
1,08017,8017
2,08940,8940
3,08192,8192
4,08003,8003
5,08005,8005
6,17411,17411
7,17196,17196
8,08002,8002
9,08001,8001


Municipalities per postal code:


municipi
1    203
2      6
Name: count, dtype: int64

codi_postal  municipi                 
8197         Barcelona                    1
             SantCugatdelVallès           1
8227         Barcelona                    1
             Terrassa                     1
8293         Collbató                     1
             SantaColomadeGramenet        1
8330         Barcelona                    1
             PremiàdeMar                  6
8902         Barcelona                    1
             HospitaletdeLlobregat(L')    1
8930         Barcelona                    1
             SantAdriàdeBesòs             2
Name: count, dtype: int64

XML prefix frequencies (no province inference):


codi_postal
08    617
17     81
25     45
43     20
28      1
00      1
Name: count, dtype: int64

In [23]:

ambiguous_postal_rows = df.loc[df["codi_postal"].isin(multiple_postal)].copy()
print("Entity/address context for postal codes with multiple municipality labels:")
with pd.option_context("display.max_colwidth", None):
    display(ambiguous_postal_rows[
        ["n_mero_registre", "nom_de_l_entitat", "adre_a", "codi_postal", "municipi"]
    ].sort_values(["codi_postal", "municipi"]))
print("Shortest literal postal values after numeric inference:")
short_numeric_postal = df["codi_postal"].lt(1000)
display(df.loc[short_numeric_postal,
               ["n_mero_registre", "nom_de_l_entitat", "codi_postal", "municipi"]]
        .assign(XML_text=xml_text.loc[short_numeric_postal, "codi_postal"]))


Entity/address context for postal codes with multiple municipality labels:


,n_mero_registre,nom_de_l_entitat,adre_a,codi_postal,municipi
595,485.000,FundaciónÁbside,"CallerBrollador,4(Valldoreix)",8197,Barcelona
247,981.000,AssociaciódeJovesAfricans/esdeCatalunya,"PasseigCamèlies,8",8197,SantCugatdelVallès
263,803.000,AssociacióEducationalDevelopmentFoundationNepal(EDFON),"CarrerBorgesBlanques,1204t1a",8227,Barcelona
133,398.000,AssociacióAlternativa3Fillsdelcafè,"CtraCastellar,526",8227,Terrassa
229,583.000,AssociaciódeCollbatóperalaSolidaritat,"PasseigdelaFumada,8",8293,Collbató
98,219.000,AsociaciónSocio-CulturalMacodouSSall,"DoctorTorrasiBages,202n1a",8293,SantaColomadeGramenet
616,885.000,FundaciónIbnBattuta,"CarrerSantPau,82",8330,Barcelona
14,348.000,AMAMESPAÑA-AsociacióndeMujeresAnti-mutilaciones,"RamonLlull,24baixos2a",8330,PremiàdeMar
198,836.000,AssociacióCooperacióActivaONGD,"CarrerReverendParareda,28,1-3",8330,PremiàdeMar
400,437.000,AssociacióperalDesenvolupamentAgrícoladeKUWONKUielseudistricte(Gambia),"CarrerMossènJacintVerdaguer,Baix1a",8330,PremiàdeMar


Shortest literal postal values after numeric inference:


,n_mero_registre,nom_de_l_entitat,codi_postal,municipi,XML_text
339,540.0,AssociacióLIVEMÈDIAperalesArtsiCulturesenXarxa,800,VilanovailaGeltrú,00800


### `codi_postal` — conclusion

- 765 complete values; 209 distinct. Every literal XML value has five digits; 618 start with zero.
- pandas inferred `int64`, losing leading zeros in the loaded representation. Literal `00800` appears as `800` in registration 540; its geographic correctness is unverified.
- Six postal codes occur with multiple municipality labels: `08197`, `08227`, `08293`, `08330`, `08902`, `08930`. Full address/name context is displayed for review.
- 21 municipality labels have multiple postal codes. These relationships are not one-to-one and alone do not prove errors.
- Candidate for `Entidad.codigo_postal` (`str | None`), not `Localidad.codigo`. No zeros are restored and no codes are changed.

### `codi_postal` — conclusión

- 765 valores completos y 209 distintos. Todos los textos XML tienen cinco dígitos; 618 empiezan por cero.
- pandas infiere `int64` y pierde ceros iniciales en la representación cargada. `00800` se observa como `800` en el registro 540; su exactitud geográfica no está comprobada.
- Seis códigos aparecen con varios municipios: `08197`, `08227`, `08293`, `08330`, `08902`, `08930`. Se muestra contexto de nombre y dirección para revisión.
- 21 municipios tienen varios códigos. La relación no es uno a uno y por sí sola no demuestra errores.
- Candidato a `Entidad.codigo_postal` (`str | None`), no `Localidad.codigo`. No se restauran ceros ni se cambian códigos.

## `municipi`

In [24]:
col = 'municipi'
s = df[col]
print("dtype:", s.dtype)
print("missing:", s.isna().sum())
print("non-null:", s.notna().sum())
print("unique:", s.nunique())
display(s.dropna().head(10))
display(s.value_counts(dropna=False).head(20))
print("Blank values:", s.str.fullmatch(r"\s*", na=False).sum())
print("Leading/trailing whitespace:", (s.notna() & s.ne(s.str.strip())).sum())
print("Whitespace anywhere:", s.str.contains(r"\s", na=False).sum())
print("Literal placeholder candidates (not recoded):")
display(s[s.isin(["_U", "NULL", "Null", "null", "None", "N/A", "-", "0"])].value_counts())

dtype: str
missing: 0
non-null: 765
unique: 113


0     SantCugatdelVallès
1              Barcelona
2    CornellàdeLlobregat
3    SantQuirzedelVallès
4              Barcelona
5              Barcelona
6               Vidreres
7                   Salt
8              Barcelona
9              Barcelona
Name: municipi, dtype: str

municipi
Barcelona                    428
Lleida                        33
Girona                        30
Terrassa                      17
SantCugatdelVallès            15
Sabadell                      13
HospitaletdeLlobregat(L')     10
Badalona                       9
Figueres                       8
MolinsdeRei                    7
PremiàdeMar                    6
Tarragona                      6
Vic                            6
CornellàdeLlobregat            5
Salt                           5
SantaPerpètuadeMogoda          5
SantaColomadeGramenet          5
Banyoles                       5
SantBoideLlobregat             4
SantJustDesvern                4
Name: count, dtype: int64

Blank values: 0
Leading/trailing whitespace: 0
Whitespace anywhere: 0
Literal placeholder candidates (not recoded):


Series([], Name: count, dtype: int64)

In [25]:
with pd.option_context("display.max_rows", None):
    display(s.value_counts())
postal_per_municipality = df.groupby(col)["codi_postal"].nunique()
display(postal_per_municipality.value_counts())
display(postal_per_municipality[postal_per_municipality.gt(1)].sort_values(ascending=False))
print("Differences from literal XML:", (s.notna() & s.ne(xml_text[col])).sum())

municipi
Barcelona                    428
Lleida                        33
Girona                        30
Terrassa                      17
SantCugatdelVallès            15
Sabadell                      13
HospitaletdeLlobregat(L')     10
Badalona                       9
Figueres                       8
MolinsdeRei                    7
PremiàdeMar                    6
Tarragona                      6
Vic                            6
CornellàdeLlobregat            5
Salt                           5
SantaPerpètuadeMogoda          5
SantaColomadeGramenet          5
Banyoles                       5
SantBoideLlobregat             4
SantJustDesvern                4
PratdeLlobregat(El)            4
Mataró                         4
Arbúcies                       4
CerdanyoladelVallès            4
Olot                           4
SantQuirzedelVallès            3
Valls                          3
Masnou(El)                     3
FranquesesdelVallès(Les)       3
Granollers                     3
R

codi_postal
1     92
2      8
3      4
5      3
45     1
7      1
6      1
10     1
4      1
8      1
Name: count, dtype: int64

municipi
Barcelona                    45
Lleida                       10
Terrassa                      8
Girona                        7
HospitaletdeLlobregat(L')     6
SantaColomadeGramenet         5
Sabadell                      5
Badalona                      5
SantCugatdelVallès            4
Mataró                        3
Reus                          3
Tarragona                     3
Granollers                    3
FranquesesdelVallès(Les)      2
CerdanyoladelVallès           2
Amposta                       2
Manresa                       2
SantPeredeRibes               2
Salt                          2
SantaPerpètuadeMogoda         2
VilanovailaGeltrú             2
Name: codi_postal, dtype: int64

Differences from literal XML: 0


In [26]:

print("Length extremes among distinct municipality labels:")
municipality_labels = df["municipi"].drop_duplicates()
municipality_lengths = municipality_labels.str.len()
municipality_profile = pd.DataFrame({"label": municipality_labels, "length": municipality_lengths})
display(municipality_profile.nsmallest(5, "length"))
display(municipality_profile.nlargest(5, "length"))
print("Original XML postal prefixes per municipality (no province assignment):")
prefix_counts = xml_text["codi_postal"].str[:2].groupby(df["municipi"]).nunique()
display(prefix_counts.value_counts().sort_index().rename_axis("distinct literal prefixes"))
multiple_prefix_labels = prefix_counts[prefix_counts.gt(1)].index
display(df.loc[df["municipi"].isin(multiple_prefix_labels),
               ["n_mero_registre", "nom_de_l_entitat", "codi_postal", "municipi"]])


Length extremes among distinct municipality labels:


,label,length
370,Vic,3
7,Salt,4
100,Teià,4
170,Gavà,4
214,Reus,4


,label,length
15,HospitaletdeLlobregat(L'),25
26,SantaMariadePalautordera,24
160,FranquesesdelVallès(Les),24
186,Palau-solitàiPlegamans,22
50,SantaPerpètuadeMogoda,21


Original XML postal prefixes per municipality (no province assignment):


distinct literal prefixes
1    112
2      1
Name: count, dtype: int64

,n_mero_registre,nom_de_l_entitat,codi_postal,municipi
339,540.000,AssociacióLIVEMÈDIAperalesArtsiCulturesenXarxa,800,VilanovailaGeltrú
344,755.000,AssociacióLonni,8800,VilanovailaGeltrú
607,1.012,FundaciónDiagramaIntervenciónPsicosocial,8800,VilanovailaGeltrú


### `municipi` — conclusion

- 765 populated values; 113 distinct labels. `Barcelona` occurs 428 times.
- No values contain whitespace. Compound labels such as `SantCugatdelVallès` join words; single-word labels such as `Barcelona` are not problematic for that reason.
- Length extremes, postal-code multiplicity and literal postal-prefix multiplicity are displayed; prefixes are not assigned to provinces.
- 112 municipality labels occur with one literal postal prefix; `VilanovailaGeltrú` occurs with two: `00` (one row, `00800`) and `08` (two rows, `08800`). This is an internal discrepancy for review; the correct postal code is not established here.
- Candidate for `Localidad.nombre` via `Entidad.en_localidad`. Locality code, province code/name and explicit province relationship are absent.
- Geographic correctness and equivalence to official municipality labels are not established.

### `municipi` — conclusión

- 765 valores informados y 113 etiquetas distintas. `Barcelona` aparece 428 veces.
- Sin caracteres de espacio. Nombres compuestos como `SantCugatdelVallès` concatenan palabras; esto no afecta a nombres de una palabra como `Barcelona`.
- Se muestran longitudes extremas y multiplicidad de códigos y prefijos postales; no se asignan provincias mediante prefijos.
- 112 municipios tienen un prefijo postal literal; `VilanovailaGeltrú` tiene dos: `00` (una fila, `00800`) y `08` (dos filas, `08800`). Es una discrepancia interna para revisión; no se establece aquí el código correcto.
- Candidato a `Localidad.nombre` mediante `Entidad.en_localidad`. Faltan código municipal, código/nombre de provincia y relación provincial explícita.
- No se establece exactitud geográfica ni equivalencia con etiquetas municipales oficiales.

## `tel_fon`

In [27]:
col = 'tel_fon'
s = df[col]
print("dtype:", s.dtype)
print("missing:", s.isna().sum())
print("non-null:", s.notna().sum())
print("unique:", s.nunique())
display(s.dropna().head(10))
display(s.value_counts(dropna=False).head(20))
print("Blank values:", s.str.fullmatch(r"\s*", na=False).sum())
print("Leading/trailing whitespace:", (s.notna() & s.ne(s.str.strip())).sum())
print("Whitespace anywhere:", s.str.contains(r"\s", na=False).sum())
print("Literal placeholder candidates (not recoded):")
display(s[s.isin(["_U", "NULL", "Null", "null", "None", "N/A", "-", "0"])].value_counts())

dtype: str
missing: 221
non-null: 544
unique: 510


0     932173600
1     932805516
2     934751735
3     937214977
4     935334238
5     932773600
7     972234686
8     934410433
10    936802048
12    937839668
Name: tel_fon, dtype: str

tel_fon
NaN          221
933188444      4
973275415      3
934589595      3
972219916      3
972405454      3
932173600      2
935334238      2
933027569      2
934159704      2
934161345      2
933026276      2
972580741      2
934443800      2
934415335      2
934817373      2
933688481      2
972691123      2
934873816      2
933011381      2
Name: count, dtype: int64

Blank values: 0
Leading/trailing whitespace: 0
Whitespace anywhere: 36
Literal placeholder candidates (not recoded):


Series([], Name: count, dtype: int64)

In [28]:
display(s.str.len().value_counts(dropna=False))
single_nine_digits = s.str.fullmatch(r"[0-9]{9}", na=False)
print("Populated cells outside single nine-digit pattern:", (s.notna() & ~single_nine_digits).sum())
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(df.loc[s.notna() & ~single_nine_digits,
                   ["n_mero_registre", "nom_de_l_entitat", col]])
print("First digit of single nine-digit cells:")
display(s[single_nine_digits].str.slice(0, 1).value_counts())
print("Populated cells with newline:", s.str.contains("\n", regex=False, na=False).sum())
print("Registrations per whole populated phone string:")
display(df.loc[s.notna()].groupby(col)["n_mero_registre"].nunique().value_counts())

tel_fon
9.0     504
NaN     221
19.0     26
20.0      5
14.0      2
21.0      1
12.0      1
11.0      1
16.0      1
26.0      1
17.0      1
18.0      1
Name: count, dtype: int64

Populated cells outside single nine-digit pattern: 40


,n_mero_registre,nom_de_l_entitat,tel_fon
20,454.0,ArquitectosTécnicosSinFronteras,934011051\n934016300
38,460.0,AsociaciónComisiónCatólicaEspañoladeMigración(ACCEM),934328526\n915327478/9
55,483.0,"AsociacióndeProsumidoresagroecológicos-AGROSOLIDARIASeccionalCatalunya""AGROSOLIDARIACATALUNYA""",932433458\n934083754
64,426.0,AsociaciónGESTA,933480958 \n933497000
85,476.0,AsociaciónparalaAccióndelDesarrolloSostenibleGlobalContralaPobrezaExtremaADESOX,936671477\n932137000
94,498.0,AsociaciónRehabiMed,932402384\n933933770
138,472.0,AssociacióAméricaEspañaSolidaridadyCooperación(AESCO),932433458\n914771000
165,422.0,"AssociacióCatalanadeCooperacióperalDesenvolupamentdelaComunitatdeBeniAhmedCharquia,ChefChaouen",933370958 \n933370000
166,536.0,AssociacióCatalanadelCàncerAssociaciód'UsuarisdelaSanitat-Càncer,932850259\n932851000
173,455.0,AssociacióCatalanaperlaInfànciaMaltractada(ACIM),935114416/17


First digit of single nine-digit cells:


tel_fon
9    481
6     22
8      1
Name: count, dtype: int64

Populated cells with newline: 34
Registrations per whole populated phone string:


n_mero_registre
1    482
2     23
3      4
4      1
Name: count, dtype: int64

In [29]:

phone_column = 'tel_fon'
phone_categories = df[phone_column].map(phone_category)
print("Mutually exclusive phone categories; multiline takes priority over extension/slash/prefix:")
display(phone_categories.value_counts().rename_axis("category"))
multiline_phones = df.loc[phone_categories.eq("multiline"), phone_column]
# Inspect line text verbatim in a diagnostic table; df cells remain intact.
phone_line_records = [
    {"row_index": row_index, "line_number": line_number,
      "literal_repr": repr(line), "characters": len(line),
      "exactly_nine_digits": bool(re.fullmatch(r"[0-9]{9}", line))}
    for row_index, value in multiline_phones.items()
    for line_number, line in enumerate(value.splitlines(), start=1)
]
phone_line_profile = pd.DataFrame(phone_line_records)
print("Multiline rows with exactly two bare nine-digit lines:",
      multiline_phones.map(lambda value: len(value.splitlines()) == 2 and
                          all(re.fullmatch(r"[0-9]{9}", line) for line in value.splitlines())).sum())
print("Lines outside the bare nine-digit pattern (spaces preserved in repr):")
display(phone_line_profile.loc[~phone_line_profile["exactly_nine_digits"]])
shared_phone_rows, shared_phone_summary = repeated_value_summary(phone_column)
print("Shared whole phone strings: identifiers and location context:")
display(shared_phone_summary)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(shared_phone_rows[
        [phone_column, "n_mero_registre", "nom_de_l_entitat", "cif_entitat", "adre_a", "municipi"]
    ].sort_values([phone_column, "n_mero_registre"]))


Mutually exclusive phone categories; multiline takes priority over extension/slash/prefix:


category
single nine digits             504
missing                        221
multiline                       34
extension, single line           4
slash, single line               1
country prefix, single line      1
Name: count, dtype: int64

Multiline rows with exactly two bare nine-digit lines: 26
Lines outside the bare nine-digit pattern (spaces preserved in repr):


,row_index,line_number,literal_repr,characters,exactly_nine_digits
3,38,2,'915327478/9',11,False
6,64,1,'933480958 ',10,False
14,165,1,'933370958 ',10,False
26,374,1,'932844392 ',10,False
32,400,1,'937593529 ',10,False
41,486,2,'933570212 Ext230',16,False
56,633,1,'933013194 ',10,False
67,747,2,'93431426',8,False


Shared whole phone strings: identifiers and location context:


,rows,registrations,names,municipalities,addresses,known_CIFs,missing_CIFs
tel_fon,,,,,,,
932173600,2,2,2,2,2,1,1
933011381,2,2,2,1,2,2,0
933026276,2,2,2,1,2,2,0
933027569,2,2,2,1,2,2,0
933188444,4,4,4,1,4,3,1
933293054,2,2,2,1,2,2,0
933688481,2,2,2,1,2,2,0
933795637,2,2,2,1,2,2,0
934159704,2,2,2,1,2,2,0


,tel_fon,n_mero_registre,nom_de_l_entitat,cif_entitat,adre_a,municipi
489,932173600,206.000,CatalunyaMiraaCamerun,NaN,"SantGermà,153er2a",Barcelona
0,932173600,306.000,180ºParaLaCooperaciónYElDesarrollo,G63749444,"PladelVinyet,9BlocDbaixos",SantCugatdelVallès
702,933011381,349.000,GrainFundacióPrivada,G59361873,"Girona,25ppal",Barcelona
425,933011381,695.000,"AssociacióRevistaSoberaníaAlimentaria,BiodiversidadyCulturas",G65692519,"CarrerGirona,25principal",Barcelona
140,933026276,128.000,AssociacióAmicsdelNepal,G60916715,"Diputació,296",Barcelona
450,933026276,362.000,AssociacióSolidaritatiComunicació-SICOM,G60789203,"Diputació,296baixos",Barcelona
27,933027569,148.000,AsociacionAfricaAmericaEuropadeRegionesyCiudades(AAERYC),G63853287,"PortaldeSantaMadrona,12pis3",Barcelona
603,933027569,149.000,FundaciónCiudadaníayBuenGobierno,G63388284,"PortaldeSantaMadrona,12planta19porta3",Barcelona
175,933188444,36.000,AssociacióCatalanaperlaPau,NaN,"ViaLaietana,161rpisdespatx114",Barcelona
176,933188444,244.000,AssociacióCatalunya-Líban,G58144759,"ViaLaietana,16,1rpis",Barcelona


### `tel_fon` — conclusion

- 544 populated strings, 221 missing (28.9%); 510 distinct populated strings.
- 504 contain exactly nine digits. The other 40 are: 34 multiline, 4 single-line extensions, 1 single-line slash notation and 1 single-line country prefix. Categories are exclusive, with multiline taking priority.
- 26 multiline cells contain exactly two bare nine-digit lines. Other lines include spaces, slash notation, an extension or an eight-digit second line; verbatim examples are displayed.
- Of the 504 single nine-digit strings, 22 start with 6. The column name does not guarantee landline service; format checks do not establish number validity.
- 28 exact strings repeat across 62 rows; 27 groups have multiple address strings and 3 span multiple municipality labels. Sharing a number is not evidence of entity identity.
- Candidate contact component for `Entidad.contacto`; source cells are not split or rewritten.

### `tel_fon` — conclusión

- 544 textos informados, 221 ausentes (28,9%) y 510 textos distintos.
- 504 contienen nueve dígitos. Los otros 40: 34 multilínea, 4 con extensión en una línea, 1 con barra en una línea y 1 con prefijo internacional en una línea. Categorías excluyentes; multilínea tiene prioridad.
- 26 celdas multilínea tienen dos líneas de nueve dígitos exactos. Otras incluyen espacios, barra, extensión o una segunda línea de ocho dígitos; se muestran literalmente.
- 22 de los 504 valores de nueve dígitos empiezan por 6. El rótulo no garantiza teléfono fijo; el formato no acredita validez del número.
- 28 textos se repiten en 62 filas; 27 grupos tienen varias direcciones y 3 varios municipios. Compartir número no demuestra identidad de entidad.
- Componente candidato para `Entidad.contacto`; no se separan ni reescriben celdas.

## `tel_fon_m_bil`

In [30]:
col = 'tel_fon_m_bil'
s = df[col]
print("dtype:", s.dtype)
print("missing:", s.isna().sum())
print("non-null:", s.notna().sum())
print("unique:", s.nunique())
display(s.dropna().head(10))
display(s.value_counts(dropna=False).head(20))
print("Blank values:", s.str.fullmatch(r"\s*", na=False).sum())
print("Leading/trailing whitespace:", (s.notna() & s.ne(s.str.strip())).sum())
print("Whitespace anywhere:", s.str.contains(r"\s", na=False).sum())
print("Literal placeholder candidates (not recoded):")
display(s[s.isin(["_U", "NULL", "Null", "null", "None", "N/A", "-", "0"])].value_counts())

dtype: str
missing: 212
non-null: 553
unique: 543


1                620030453
2                660300281
3     650399867\n629401111
9                629031708
11               678218160
12               600569073
13               638736625
14               696733522
15               658577264
16               639424278
Name: tel_fon_m_bil, dtype: str

tel_fon_m_bil
NaN                     212
654812817                 2
632291959                 2
648761199                 2
606995623                 2
602881484                 2
657317340                 2
676664757                 2
630659083                 2
656801438                 2
650918378                 2
620030453                 1
660300281                 1
650399867\n629401111      1
629031708                 1
678218160                 1
600569073                 1
638736625                 1
696733522                 1
658577264                 1
Name: count, dtype: int64

Blank values: 0
Leading/trailing whitespace: 0
Whitespace anywhere: 23
Literal placeholder candidates (not recoded):


Series([], Name: count, dtype: int64)

In [31]:
display(s.str.len().value_counts(dropna=False))
single_nine_digits = s.str.fullmatch(r"[0-9]{9}", na=False)
print("Populated cells outside single nine-digit pattern:", (s.notna() & ~single_nine_digits).sum())
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(df.loc[s.notna() & ~single_nine_digits,
                   ["n_mero_registre", "nom_de_l_entitat", col]])
print("First digit of single nine-digit cells:")
display(s[single_nine_digits].str.slice(0, 1).value_counts())
print("Populated cells with newline:", s.str.contains("\n", regex=False, na=False).sum())
print("Registrations per whole populated phone string:")
display(df.loc[s.notna()].groupby(col)["n_mero_registre"].nunique().value_counts())

tel_fon_m_bil
9.0     530
NaN     212
19.0     22
22.0      1
Name: count, dtype: int64

Populated cells outside single nine-digit pattern: 23


,n_mero_registre,nom_de_l_entitat,tel_fon_m_bil
3,318.0,AccióSolidàriadelVallès,650399867\n629401111
30,563.0,AsociaciónAmigosdeSilva,687724942\n607880830
36,475.0,AsociaciónChelonia,646578931\n650931000
57,640.0,AsociacióndeVoluntariosChildrenofÁfrica,661879913\n620531021
77,560.0,AsociaciónMujeresBurkina,670719832\n609819964
85,476.0,AsociaciónparalaAccióndelDesarrolloSostenibleGlobalContralaPobrezaExtremaADESOX,629930636\n651799582
138,472.0,AssociacióAméricaEspañaSolidaridadyCooperación(AESCO),654389298\n699900000
204,634.0,AssociacióCoumbaKandepelDesenvolupamentRural,659201839\n673232922
208,488.0,AssociacióCulturalAfricano-Catalanad'ajudaalCodesenvolupament(ACACAC),665267435\n656911000
239,346.0,AssociaciódeDonesdeKerewanSambaSirapelDesenvolupamentdeGirona,619958076\n632287537


First digit of single nine-digit cells:


tel_fon_m_bil
6    516
9      8
7      6
Name: count, dtype: int64

Populated cells with newline: 23
Registrations per whole populated phone string:


n_mero_registre
1    533
2     10
Name: count, dtype: int64

In [32]:
display(pd.crosstab(df["tel_fon"].isna(), df["tel_fon_m_bil"].isna(),
                    rownames=["Telephone missing"], colnames=["Mobile missing"]))
same_phone = df["tel_fon"].notna() & df["tel_fon"].eq(df["tel_fon_m_bil"])
print("Equal populated phone strings:", same_phone.sum())
display(df.loc[same_phone, ["n_mero_registre", "nom_de_l_entitat", "tel_fon", "tel_fon_m_bil"]])

Mobile missing,False,True
Telephone missing,,
False,344,200
True,209,12


Equal populated phone strings: 1


,n_mero_registre,nom_de_l_entitat,tel_fon,tel_fon_m_bil
465,781.0,AssociacióUnimCultures,600887640,600887640


In [33]:

phone_column = 'tel_fon_m_bil'
phone_categories = df[phone_column].map(phone_category)
print("Mutually exclusive phone categories; multiline takes priority over extension/slash/prefix:")
display(phone_categories.value_counts().rename_axis("category"))
multiline_phones = df.loc[phone_categories.eq("multiline"), phone_column]
# Inspect line text verbatim in a diagnostic table; df cells remain intact.
phone_line_records = [
    {"row_index": row_index, "line_number": line_number,
      "literal_repr": repr(line), "characters": len(line),
      "exactly_nine_digits": bool(re.fullmatch(r"[0-9]{9}", line))}
    for row_index, value in multiline_phones.items()
    for line_number, line in enumerate(value.splitlines(), start=1)
]
phone_line_profile = pd.DataFrame(phone_line_records)
print("Multiline rows with exactly two bare nine-digit lines:",
      multiline_phones.map(lambda value: len(value.splitlines()) == 2 and
                          all(re.fullmatch(r"[0-9]{9}", line) for line in value.splitlines())).sum())
print("Lines outside the bare nine-digit pattern (spaces preserved in repr):")
display(phone_line_profile.loc[~phone_line_profile["exactly_nine_digits"]])
shared_phone_rows, shared_phone_summary = repeated_value_summary(phone_column)
print("Shared whole phone strings: identifiers and location context:")
display(shared_phone_summary)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(shared_phone_rows[
        [phone_column, "n_mero_registre", "nom_de_l_entitat", "cif_entitat", "adre_a", "municipi"]
    ].sort_values([phone_column, "n_mero_registre"]))

both_phone_columns = ["tel_fon", "tel_fon_m_bil"]
both_populated = df[both_phone_columns].notna().all(axis=1)
same_phone_pair = both_populated & df.duplicated(both_phone_columns, keep=False)
print("Rows sharing both populated telephone strings:", same_phone_pair.sum())
display(df.loc[same_phone_pair,
               ["n_mero_registre", "nom_de_l_entitat", "cif_entitat", "municipi"] + both_phone_columns])


Mutually exclusive phone categories; multiline takes priority over extension/slash/prefix:


category
single nine digits    530
missing               212
multiline              23
Name: count, dtype: int64

Multiline rows with exactly two bare nine-digit lines: 22
Lines outside the bare nine-digit pattern (spaces preserved in repr):


,row_index,line_number,literal_repr,characters,exactly_nine_digits
37,623,2,'+34690692228',12,False


Shared whole phone strings: identifiers and location context:


,rows,registrations,names,municipalities,addresses,known_CIFs,missing_CIFs
tel_fon_m_bil,,,,,,,
602881484,2,2,2,1,2,2,0
606995623,2,2,2,1,2,0,2
630659083,2,2,2,1,2,2,0
632291959,2,2,2,2,2,2,0
648761199,2,2,2,1,2,1,1
650918378,2,2,2,2,2,2,0
654812817,2,2,2,2,2,2,0
656801438,2,2,2,1,2,2,0
657317340,2,2,2,2,2,2,0


,tel_fon_m_bil,n_mero_registre,nom_de_l_entitat,cif_entitat,adre_a,municipi
223,602881484,648.000,Associaciód'ajudasolidàriaMaliVert,G55135487,"CarrerdelaBarca,nº7",Banyoles
436,602881484,714.000,AssociaciósocioculturaldedonesafricanesdeBanyolesLegkiYakaru,G17914284,"CarrerBarcelona,52baixos(CentreCívic)",Banyoles
151,606995623,215.000,"AssociacióAvalua,AnàlisiiDesenvolupament",NaN,"Rocafort,242bisppal",Barcelona
457,606995623,676.000,AssociacióTaulaCatalanadeCodesenvolupament,NaN,"Rocafort,242bis3r",Barcelona
376,630659083,782.000,AssociacióNúriaSocial,G55250005,"CarrerdelCarme,nº8",Olot
521,630659083,793.000,CooperativaResilienceEarthSCCL,F55279665,"CarrerdelCarme,81",Olot
31,632291959,654.000,AsociaciónBissauGuineanosdeCataluña,G61581856,CentreCívicSantMartí-CarrerSelvadeMarnº215,Barcelona
552,632291959,720.000,FederacióndelasAsociacionesdeAfricaNegradeSabadellyCatalunya(FEDEAFRICANESCA),G55258909,"CarrerSantJoan,20baixos",Sabadell
111,648761199,111.000,AssociacióAccióEcologistaViureSenseNuclears,NaN,"SantSalvador,96baixos",Barcelona
547,648761199,865.000,Federaciód’EcologistesenAcciódeCatalunya,G62110325,"CarrerOlzinelles,31",Barcelona


Rows sharing both populated telephone strings: 6


,n_mero_registre,nom_de_l_entitat,cif_entitat,municipi,tel_fon,tel_fon_m_bil
376,782.0,AssociacióNúriaSocial,G55250005,Olot,972691123,630659083
481,34.0,AssociacióYamunad'AjutalaInfància,G62352042,Barcelona,934196944,656801438
521,793.0,CooperativaResilienceEarthSCCL,F55279665,Olot,972691123,630659083
665,860.0,FundacióPrivadaGentis,G17679267,Barcelona,972405454,650918378
680,763.0,FundacióPrivadaResilis,G17925215,Girona,972405454,650918378
697,748.0,FundacióYamuna,G64261209,Barcelona,934196944,656801438


### `tel_fon_m_bil` — conclusion

- 553 populated strings, 212 missing (27.7%); 543 distinct populated strings.
- 530 contain exactly nine digits; 23 are multiline. Of those 23, 22 contain two bare nine-digit lines and one includes a `+34` prefix on its second line.
- Eight single nine-digit strings start with 9; mobile service is not established by the column label.
- 10 exact strings repeat across 20 rows; all groups have different addresses and 4 span multiple municipality labels.
- Both phone fields are missing in 12 rows. One row has equal populated values in both fields; three two-row groups share the same populated phone pair but have different CIFs.
- Candidate component for `Entidad.contacto`; no telephone selection or splitting rule is applied.

### `tel_fon_m_bil` — conclusión

- 553 textos informados, 212 ausentes (27,7%) y 543 textos distintos.
- 530 contienen nueve dígitos; 23 son multilínea. De estas, 22 tienen dos líneas de nueve dígitos y una incluye `+34` en la segunda línea.
- Ocho valores de nueve dígitos empiezan por 9; el rótulo no acredita servicio móvil.
- 10 textos se repiten en 20 filas; todos los grupos tienen direcciones distintas y 4 varios municipios.
- En 12 filas faltan ambos teléfonos. Una fila tiene el mismo valor informado en ambos campos; tres grupos de dos filas comparten la pareja telefónica, pero tienen CIF distintos.
- Componente candidato para `Entidad.contacto`; no se seleccionan ni separan teléfonos.

## `correu_electr_nic`

In [34]:
col = 'correu_electr_nic'
s = df[col]
print("dtype:", s.dtype)
print("missing:", s.isna().sum())
print("non-null:", s.notna().sum())
print("unique:", s.nunique())
display(s.dropna().head(10))
display(s.value_counts(dropna=False).head(20))
print("Blank values:", s.str.fullmatch(r"\s*", na=False).sum())
print("Leading/trailing whitespace:", (s.notna() & s.ne(s.str.strip())).sum())
print("Whitespace anywhere:", s.str.contains(r"\s", na=False).sum())
print("Literal placeholder candidates (not recoded):")
display(s[s.isin(["_U", "NULL", "Null", "null", "None", "N/A", "-", "0"])].value_counts())

dtype: str
missing: 35
non-null: 730
unique: 726


1                     laia@1x1microcreditorg
2                     palomes@periodistesorg
3              acciosolidariavalles@gmailcom
4     ingman@ingmancatmaria_novella@igmancat
5                        avallejo@nospetites
6                         aafrowest@gmailcom
7                 agermanamentperu@gmail.com
8               agermanament@agermanamentorg
9               coordinadora@agoranordsudorg
11              alegriasinfronteras@gmailcom
Name: correu_electr_nic, dtype: str

correu_electr_nic
NaN                                       35
cooperacioactivatgn@gmailcom               2
dialladiarra@hotmailcom                    2
josep@talaiainfo                           2
info@yamunaorg                             2
laia@1x1microcreditorg                     1
palomes@periodistesorg                     1
acciosolidariavalles@gmailcom              1
ingman@ingmancatmaria_novella@igmancat     1
avallejo@nospetites                        1
aafrowest@gmailcom                         1
agermanamentperu@gmail.com                 1
agermanament@agermanamentorg               1
coordinadora@agoranordsudorg               1
alegriasinfronteras@gmailcom               1
ongdase@gmailcom                           1
afdspain@gmailcom                          1
amamespana@yahooes                         1
amazancat@hotmailcom                       1
informacio@amazoniacat                     1
Name: count, dtype: int64

Blank values: 0
Leading/trailing whitespace: 0
Whitespace anywhere: 9
Literal placeholder candidates (not recoded):


Series([], Name: count, dtype: int64)

In [35]:
display(s.str.len().describe())
display(s.str.count("@").value_counts(dropna=False))
email_pattern = r"[^\s@]+@[^\s@]+\.[^\s@]+"
single_email = s.str.fullmatch(email_pattern, na=False)
print("Simple single-address pattern matches:", single_email.sum())
print("Populated cells without any dot:", (s.notna() & ~s.str.contains(".", regex=False, na=False)).sum())
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(df.loc[s.notna() & s.str.count("@").ne(1),
                   ["n_mero_registre", "nom_de_l_entitat", col]])
    display(df.loc[s.notna() & ~single_email, ["n_mero_registre", col]].head(20))
print("Registrations per whole email string:")
display(df.loc[s.notna()].groupby(col)["n_mero_registre"].nunique().value_counts())
contact_columns = ["tel_fon", "tel_fon_m_bil", col]
no_contact = df[contact_columns].isna().all(axis=1)
print("Rows without any phone or email:", no_contact.sum())
display(df.loc[no_contact, ["n_mero_registre", "nom_de_l_entitat"] + contact_columns + ["web"]])
# Domain-specific check restricted to cells with exactly one @.
single_at = s.notna() & s.str.count("@").eq(1)
email_domains = s.loc[single_at].str.split("@").str[-1]
domain_without_dot = ~email_domains.str.contains(".", regex=False, na=False)
print("Populated cells with exactly one @:", single_at.sum())
print("Their domain parts without any dot:", domain_without_dot.sum())
with pd.option_context("display.max_colwidth", None):
    display(df.loc[email_domains.index[domain_without_dot],
                   ["n_mero_registre", col]].head(10))
print("Single-@ email values match literal XML:",
      s.loc[single_at].eq(xml_text.loc[single_at, col]).all())

count    730.000000
mean      22.724658
std        8.308164
min       10.000000
25%       18.000000
50%       21.000000
75%       26.000000
max      100.000000
Name: correu_electr_nic, dtype: float64

correu_electr_nic
1.0    703
NaN     35
2.0     21
3.0      3
0.0      2
4.0      1
Name: count, dtype: int64

Simple single-address pattern matches: 76
Populated cells without any dot: 650


,n_mero_registre,nom_de_l_entitat,correu_electr_nic
4,459.0,AccióSolidàriaIgman,ingman@ingmancatmaria_novella@igmancat
31,654.0,AsociaciónBissauGuineanosdeCataluña,agbccatalunia@gmail.com\nbubabonifaciofernandes@gmail.com
38,460.0,AsociaciónComisiónCatólicaEspañoladeMigración(ACCEM),"catalunya@accemes,administracion@accemes"
44,549.0,AsociaciónCulturalEcuatorianaporlosDerechosylaIgualdadenCataluñaAndrésRecalde-ACEDICAR,patypall@yahooesacedicar@yahooes
96,702.0,AsociaciónSakadawa,sakadawaorggmailcom
138,472.0,AssociacióAméricaEspañaSolidaridadyCooperación(AESCO),catalunyaaesco@ong-aescoorgcooperacion@ong-aescoorgdyaryas@yahoocom
170,424.0,"AssociacióCatalanaperalaCooperació,l'Esport,elDesenvolupamentil'Educació(ACarrerCarrerEDE)",rosamariafernandez@accedecatferlabella@icabcat
191,198.0,AssociacióComissióCatalanad'AjudaalRefugiat,ccar@ccar-cearorg;coordcata@ceares
199,471.0,AssociacióCooperacióGrupSuportConVos(Antiga:GrupdeSuportalColectivodeMujeresdeMatagalpa),grupsuport@yahoo.es\naraysiempre@gmail.com
234,400.0,AssociaciódeCooperacióInternacionalNord-sud(CONOSUD),ongconosud@gmail.com\ninfoconosud@gmailcom


,n_mero_registre,correu_electr_nic
1,535.0,laia@1x1microcreditorg
2,379.0,palomes@periodistesorg
3,318.0,acciosolidariavalles@gmailcom
4,459.0,ingman@ingmancatmaria_novella@igmancat
5,280.0,avallejo@nospetites
6,861.0,aafrowest@gmailcom
8,41.0,agermanament@agermanamentorg
9,336.0,coordinadora@agoranordsudorg
11,688.0,alegriasinfronteras@gmailcom
12,466.0,ongdase@gmailcom


...

Rows without any phone or email: 1


,n_mero_registre,nom_de_l_entitat,tel_fon,tel_fon_m_bil,correu_electr_nic,web
273,392.0,AssociacióEsperitHumanitarid'Espanyaal'Àfrica-...,NaN,NaN,NaN,NaN


Populated cells with exactly one @: 703
Their domain parts without any dot: 627


,n_mero_registre,correu_electr_nic
1,535.0,laia@1x1microcreditorg
2,379.0,palomes@periodistesorg
3,318.0,acciosolidariavalles@gmailcom
5,280.0,avallejo@nospetites
6,861.0,aafrowest@gmailcom
8,41.0,agermanament@agermanamentorg
9,336.0,coordinadora@agoranordsudorg
11,688.0,alegriasinfronteras@gmailcom
12,466.0,ongdase@gmailcom
13,354.0,afdspain@gmailcom


Single-@ email values match literal XML: True


In [36]:

email_categories = df["correu_electr_nic"].map(email_category)
print("Mutually exclusive whole-cell email categories:")
display(email_categories.value_counts().rename_axis("category"))
multiple_at = df["correu_electr_nic"].str.count("@").gt(1).fillna(False)
email_multiline = df["correu_electr_nic"].str.contains("\n", regex=False, na=False)
print("Multiple-@ cells with newline:", (multiple_at & email_multiline).sum())
print("Multiple-@ cells without any whitespace:",
      (multiple_at & ~df["correu_electr_nic"].str.contains(r"\s", na=False)).sum())
print("Zero-@ examples, verbatim:")
display(df.loc[email_categories.eq("no @"),
               ["n_mero_registre", "nom_de_l_entitat", "correu_electr_nic"]])
email_rows, email_shared_summary = repeated_value_summary("correu_electr_nic")
print("Repeated email strings: entity/address context:")
display(email_shared_summary)
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(email_rows.sort_values(["correu_electr_nic", "n_mero_registre"]))


Mutually exclusive whole-cell email categories:


category
single @, domain without dot     627
simple single-address pattern     76
missing                           35
multiple @                        25
no @                               2
Name: count, dtype: int64

Multiple-@ cells with newline: 9
Multiple-@ cells without any whitespace: 16
Zero-@ examples, verbatim:


,n_mero_registre,nom_de_l_entitat,correu_electr_nic
96,702.0,AsociaciónSakadawa,sakadawaorggmailcom
417,190.0,AssociacióProInfants(AntigaAssociacióperal'aju...,infoarrobaproinfantsorg


Repeated email strings: entity/address context:


,rows,registrations,names,municipalities,addresses,known_CIFs,missing_CIFs
correu_electr_nic,,,,,,,
cooperacioactivatgn@gmailcom,2,2,2,2,2,2,0
dialladiarra@hotmailcom,2,2,2,1,2,2,0
info@yamunaorg,2,2,2,1,2,2,0
josep@talaiainfo,2,2,2,1,2,1,1


,n_mero_registre,nom_de_l_entitat,naturalesa,cif_entitat,adre_a,codi_postal,municipi,tel_fon,tel_fon_m_bil,correu_electr_nic,web
197,607.0,AssociacióCooperacióActivaalCampdeTarragona,Associació,G55620439,"CarrerAmposta,34t1a",43840,Salou,NaN,630831001,cooperacioactivatgn@gmailcom,http://cooperacioactivacamptarragonawordpresscom
553,834.0,FederacióXarxadeCooperacióalDesenvolupamentdelSuddeCatalunya,Federació,G72637994,"CarrerAugust,26,3er-2a",43003,Tarragona,NaN,651191205,cooperacioactivatgn@gmailcom,http://wwwxcdcat
223,648.0,Associaciód'ajudasolidàriaMaliVert,Associació,G55135487,"CarrerdelaBarca,nº7",17820,Banyoles,972584236,602881484,dialladiarra@hotmailcom,NaN
436,714.0,AssociaciósocioculturaldedonesafricanesdeBanyolesLegkiYakaru,Associació,G17914284,"CarrerBarcelona,52baixos(CentreCívic)",17820,Banyoles,972580741,602881484,dialladiarra@hotmailcom,http://legkiyakaruwordpresscom
481,34.0,AssociacióYamunad'AjutalaInfància,Associació,G62352042,"París,97,local2",8029,Barcelona,934196944,656801438,info@yamunaorg,http://wwwyamunaorg
697,748.0,FundacióYamuna,Fundació,G64261209,"CarrerdeParís,97local-2",8029,Barcelona,934196944,656801438,info@yamunaorg,http://wwwyamunaorg
399,227.0,AssociacióperalaSalutIntegralielDesenvolupamentHumà(ASIDH),Associació,G60605110,"Maignon,12sobreàtic1A",8024,Barcelona,932640356\n934901675,NaN,josep@talaiainfo,NaN
597,228.0,"FundaciónASIDH,paralaSaludIntegralyelDesarrolloHumano",Fundació,NaN,"AvDeXile,426è1a",8028,Barcelona,932640356,NaN,josep@talaiainfo,NaN


### `correu_electr_nic` — conclusion

- 730 populated strings, 35 missing (4.6%); 726 distinct populated strings.
- Exclusive whole-cell categories: 76 match the simple single-address pattern, 627 have one `@` and no dot in the domain, 25 have multiple `@`, and 2 have no `@`.
- The 627 domain cases include `acciosolidariavalles@gmailcom`. Separately, 650 whole cells contain no dot anywhere; the two counts measure different properties.
- Of the 25 multiple-@ cells, 9 contain newlines and 16 lack whitespace. Zero-@ values are `sakadawaorggmailcom` and `infoarrobaproinfantsorg`. These forms are already in XML.
- Four exact strings repeat across eight rows; each pair has different names and address strings. Three pairs have two different known CIFs; one pair has a missing CIF. One pair spans two municipalities.
- Pattern failure does not establish deliverability or the intended address. Candidate for `Entidad.contacto`; no dots or separators are inserted.

### `correu_electr_nic` — conclusión

- 730 textos informados, 35 ausentes (4,6%) y 726 textos distintos.
- Categorías excluyentes por celda: 76 cumplen el patrón simple de correo único; 627 tienen una `@` y dominio sin punto; 25 tienen varias `@`; 2 no tienen `@`.
- Entre los 627 casos está `acciosolidariavalles@gmailcom`. Por separado, 650 celdas no contienen ningún punto en todo el texto; son medidas distintas.
- De las 25 celdas con varias `@`, 9 contienen saltos de línea y 16 carecen de espacios. Sin `@`: `sakadawaorggmailcom` e `infoarrobaproinfantsorg`. Ya aparecen así en XML.
- Cuatro textos se repiten en ocho filas; cada pareja tiene nombres y direcciones distintos. Tres parejas tienen dos CIF conocidos distintos; en una falta un CIF. Una pareja abarca dos municipios.
- El patrón no determina entrega ni dirección prevista. Candidato a `Entidad.contacto`; no se insertan puntos ni separadores.

## `web`

In [37]:
col = 'web'
s = df[col]
print("dtype:", s.dtype)
print("missing:", s.isna().sum())
print("non-null:", s.notna().sum())
print("unique:", s.nunique())
display(s.dropna().head(10))
display(s.value_counts(dropna=False).head(20))
print("Blank values:", s.str.fullmatch(r"\s*", na=False).sum())
print("Leading/trailing whitespace:", (s.notna() & s.ne(s.str.strip())).sum())
print("Whitespace anywhere:", s.str.contains(r"\s", na=False).sum())
print("Literal placeholder candidates (not recoded):")
display(s[s.isin(["_U", "NULL", "Null", "null", "None", "N/A", "-", "0"])].value_counts())

dtype: str
missing: 154
non-null: 611
unique: 607


0                 http://ca180gradosinfo/
1             http://www1x1microcreditorg
2                  http://wwweltriangleeu
4                      http://wwwigmancat
5                http://acoescatalunyaorg
7     http://agermanamentperuwordpresscom
8               http://wwwagermanamentorg
11       http://wwwalegriasinfronterasorg
12                    http://wwwaseongorg
13                  http://wwwafdspaincom
Name: web, dtype: str

web
NaN                                    154
http://wwwsolidariesorg                  3
http://wwwyamunaorg                      2
http://wwwquepoorg                       2
http://ca180gradosinfo/                  1
http://www1x1microcreditorg              1
http://wwweltriangleeu                   1
http://wwwigmancat                       1
http://acoescatalunyaorg                 1
http://agermanamentperuwordpresscom      1
http://wwwagermanamentorg                1
http://wwwalegriasinfronterasorg         1
http://wwwaseongorg                      1
http://wwwafdspaincom                    1
http://wwwamames                         1
http://wwwamazoniacat                    1
http://wwwamigosdetanzaniaorg            1
http://wwwamizadejamorg                  1
http://wwwatsfes                         1
http://wwwarxiversorg                    1
Name: count, dtype: int64

Blank values: 0
Leading/trailing whitespace: 0
Whitespace anywhere: 0
Literal placeholder candidates (not recoded):


Series([], Name: count, dtype: int64)

In [38]:
# Parsing for diagnostics only; no HTTP requests and no URL rewriting.
web_profile = pd.DataFrame({"web": s.dropna()})
web_profile["scheme"] = web_profile["web"].map(lambda value: urlsplit(value).scheme)
web_profile["hostname"] = web_profile["web"].map(lambda value: urlsplit(value).hostname)
web_profile["hostname_has_dot"] = web_profile["hostname"].map(lambda host: "." in (host or ""))
display(web_profile["scheme"].value_counts())
display(web_profile["hostname_has_dot"].value_counts())
with pd.option_context("display.max_colwidth", None):
    display(web_profile.head(20))
    display(df.loc[s.notna() & s.duplicated(keep=False),
                   ["n_mero_registre", "nom_de_l_entitat", col]].sort_values(col))
print("Differences from literal XML:", (s.notna() & s.ne(xml_text[col])).sum())

scheme
http    611
Name: count, dtype: int64

hostname_has_dot
False    611
Name: count, dtype: int64

,web,scheme,hostname,hostname_has_dot
0,http://ca180gradosinfo/,http,ca180gradosinfo,False
1,http://www1x1microcreditorg,http,www1x1microcreditorg,False
2,http://wwweltriangleeu,http,wwweltriangleeu,False
4,http://wwwigmancat,http,wwwigmancat,False
5,http://acoescatalunyaorg,http,acoescatalunyaorg,False
7,http://agermanamentperuwordpresscom,http,agermanamentperuwordpresscom,False
8,http://wwwagermanamentorg,http,wwwagermanamentorg,False
11,http://wwwalegriasinfronterasorg,http,wwwalegriasinfronterasorg,False
12,http://wwwaseongorg,http,wwwaseongorg,False
13,http://wwwafdspaincom,http,wwwafdspaincom,False


,n_mero_registre,nom_de_l_entitat,web
520,743.0,"CooperativaQuepo,SCarrerCarrerL",http://wwwquepoorg
632,324.0,FundaciónPrivadaQuepo,http://wwwquepoorg
289,124.0,AssociacióGrupd'EmpordanesosiEmpordanesesperlasolidaritat,http://wwwsolidariesorg
438,123.0,AssociacióSòcio-culturalKawralFuladu,http://wwwsolidariesorg
544,133.0,FederacióCoordinadorad’ONGSolidàriesdelescomarquesgironinesil’AltMaresme(Antiga:AssociacióCentredeRecursosperalaPauilaSolidaritat(CeDRe),http://wwwsolidariesorg
481,34.0,AssociacióYamunad'AjutalaInfància,http://wwwyamunaorg
697,748.0,FundacióYamuna,http://wwwyamunaorg


Differences from literal XML: 0


In [39]:

# A repeated scheme changes how urlsplit interprets the hostname.
scheme_repetition = df["web"].str.count(r"https?://").gt(1).fillna(False)
print("Populated cells with more than one http(s) scheme:", scheme_repetition.sum())
print("Scheme repetition is independent of the dot-in-host diagnostic.")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(df.loc[scheme_repetition, ["n_mero_registre", "web"]])
web_rows, web_shared_summary = repeated_value_summary("web")
print("Shared web strings: entity/address context:")
display(web_shared_summary)
with pd.option_context("display.max_colwidth", None, "display.max_columns", None):
    display(web_rows.sort_values(["web", "n_mero_registre"]))
all_contact_columns = ["tel_fon", "tel_fon_m_bil", "correu_electr_nic", "web"]
complete_contact = df[all_contact_columns].notna().all(axis=1)
identical_contacts = complete_contact & df.duplicated(all_contact_columns, keep=False)
print("Rows sharing all four populated contact fields:", identical_contacts.sum())
display(df.loc[identical_contacts])


Populated cells with more than one http(s) scheme: 56
Scheme repetition is independent of the dot-in-host diagnostic.


,n_mero_registre,web
28,899.000,http://https://alasgambia@org
39,926.000,http://https://cecodees/
72,911.000,http://https://masretinawixsitecom/misitio-1
73,958.000,http://https://mavasrilankaorg/
76,917.000,http://https://wwwmujerdiasporacom·
88,1.018,http://https://pstangaranacom/ca/
115,1.039,http://https://wwwaccionatorg/
117,73.000,http://https://wwwaslong/es/inicio/
128,492.000,http://https://ajudamaviurewordpresscom/about/
135,784.000,http://https://ambdretsorg/


Shared web strings: entity/address context:


,rows,registrations,names,municipalities,addresses,known_CIFs,missing_CIFs
web,,,,,,,
http://wwwquepoorg,2,2,2,1,2,2,0
http://wwwsolidariesorg,3,3,3,2,3,2,1
http://wwwyamunaorg,2,2,2,1,2,2,0


,n_mero_registre,nom_de_l_entitat,naturalesa,cif_entitat,adre_a,codi_postal,municipi,tel_fon,tel_fon_m_bil,correu_electr_nic,web
632,324.0,FundaciónPrivadaQuepo,Fundació,G64564057,"CarrerLlibertat,47,baixos",8012,Barcelona,931805768,NaN,info@quepo.org,http://wwwquepoorg
520,743.0,"CooperativaQuepo,SCarrerCarrerL",Cooperativa,F67133058,CarrerLlibertat47baixos,8012,Barcelona,931867511,NaN,info@quepoorg,http://wwwquepoorg
438,123.0,AssociacióSòcio-culturalKawralFuladu,Associació,NaN,"Peixos,5baixos",17600,Figueres,972675240,631331902,kawralfuladu@hotmailcom,http://wwwsolidariesorg
289,124.0,AssociacióGrupd'EmpordanesosiEmpordanesesperlasolidaritat,Associació,G17446881,"SantVicenç,301erpis",17600,Figueres,972677194,NaN,gees@solidariesorg,http://wwwsolidariesorg
544,133.0,FederacióCoordinadorad’ONGSolidàriesdelescomarquesgironinesil’AltMaresme(Antiga:AssociacióCentredeRecursosperalaPauilaSolidaritat(CeDRe),Federació,G17168964,"MestreFrancescCivil,3",17005,Girona,972219916,636319202,solidaries@solidariesorg,http://wwwsolidariesorg
481,34.0,AssociacióYamunad'AjutalaInfància,Associació,G62352042,"París,97,local2",8029,Barcelona,934196944,656801438,info@yamunaorg,http://wwwyamunaorg
697,748.0,FundacióYamuna,Fundació,G64261209,"CarrerdeParís,97local-2",8029,Barcelona,934196944,656801438,info@yamunaorg,http://wwwyamunaorg


Rows sharing all four populated contact fields: 2


,n_mero_registre,nom_de_l_entitat,naturalesa,cif_entitat,adre_a,codi_postal,municipi,tel_fon,tel_fon_m_bil,correu_electr_nic,web
481,34.0,AssociacióYamunad'AjutalaInfància,Associació,G62352042,"París,97,local2",8029,Barcelona,934196944,656801438,info@yamunaorg,http://wwwyamunaorg
697,748.0,FundacióYamuna,Fundació,G64261209,"CarrerdeParís,97local-2",8029,Barcelona,934196944,656801438,info@yamunaorg,http://wwwyamunaorg


### `web` — conclusion

- 611 populated strings, 154 missing (20.1%); 607 distinct populated strings.
- 56 cells contain repeated scheme text, such as `http://https://cecodees/`. For these, `urlsplit` reports `https` or `http` as hostname; that result is a parsing artifact of the source syntax.
- All 611 parsed hostnames lack dots; this includes the repeated-scheme cases. Ordinary examples such as `http://wwwigmancat` also lack a dot in the literal host text. These observations do not establish the intended domains.
- Three exact URL strings repeat across seven rows. Quepo and Yamuna pairs have different known CIFs; `http://wwwsolidariesorg` spans three names and two municipalities, with one CIF missing.
- Two Yamuna records share all four populated contact fields but have different registrations, names, CIFs, legal forms and address strings. This supports review for a possible relationship, not automatic merging.
- Candidate for `Entidad.url`; no scheme/domain repair, DNS lookup or availability check is performed.

### `web` — conclusión

- 611 textos informados, 154 ausentes (20,1%) y 607 textos distintos.
- 56 celdas repiten el esquema, como `http://https://cecodees/`. En ellas `urlsplit` devuelve `https` o `http` como hostname: efecto de análisis de la sintaxis de origen.
- Los 611 hostnames analizados carecen de puntos, incluidos esos 56 casos. Ejemplos ordinarios como `http://wwwigmancat` también carecen de punto en el host literal. No se establecen dominios previstos.
- Tres URLs se repiten en siete filas. Quepo y Yamuna tienen CIF conocidos distintos; `http://wwwsolidariesorg` abarca tres nombres y dos municipios, con un CIF ausente.
- Dos registros Yamuna comparten los cuatro contactos informados, pero difieren en registro, nombre, CIF, naturaleza y texto de dirección. Sugiere revisar una posible relación, no fusionar automáticamente.
- Candidato a `Entidad.url`; no se reparan esquemas/dominios ni se comprueba DNS o disponibilidad.

## Joint missingness / Ausencias conjuntas

In [40]:

missing_fields = ["cif_entitat", "tel_fon", "tel_fon_m_bil", "correu_electr_nic", "web"]
missing_patterns = df[missing_fields].isna().value_counts().rename("rows").to_frame()
print("Exact joint missingness patterns (True means absent):")
display(missing_patterns)
no_phone = df[["tel_fon", "tel_fon_m_bil"]].isna().all(axis=1)
no_direct_contact = no_phone & df["correu_electr_nic"].isna()
print("Email versus both-phone availability:")
display(pd.crosstab(df["correu_electr_nic"].isna(), no_phone,
                    rownames=["Email missing"], colnames=["Both phones missing"]))
print("Rows with neither phone nor email:")
display(df.loc[no_direct_contact])
print("Missing rates by CIF availability; descriptive association only:")
cif_group_sizes = df.groupby(df["cif_entitat"].isna()).size().rename("rows")
by_cif_missing = df[missing_fields[1:]].isna().groupby(df["cif_entitat"].isna()).mean().mul(100).round(1)
display(cif_group_sizes.to_frame().join(by_cif_missing.add_suffix(" missing %")))
print("CIF absence is not identical to absence of any direct-contact field:")
display(pd.crosstab(df["cif_entitat"].isna(),
                    df[["tel_fon", "tel_fon_m_bil", "correu_electr_nic"]].isna().any(axis=1),
                    rownames=["CIF missing"], colnames=["Any direct-contact field missing"]))


Exact joint missingness patterns (True means absent):


rows
cif_entitat tel_fon tel_fon_m_bil correu_electr_nic web        
False       False   False         False             False   239
                    True          False             False   149
            True    False         False             False   145
            False   False         False             True     48
            True    False         False             True     46
True        False   False         False             False    28
                    True          False             False    19
False       False   True          False             True     19
                    False         True              False    10
True        False   False         False             True      9
                    True          False             True      8
False       True    True          False             False     7
True        True    False         False             False     7
False       False   False         True              True      6
            True    False         True              True      4
                    True          False             True      3
True        True    False         True              True      3
False       False   True          True              True      2
True        False   False         True              True      2
                                                    False     2
False       True    False         True              False     2
True        True    False         False             True      2
False       False   True          True              False     1
True        False   True          True              True      1
            True    True          False             False     1
False       True    True          True              True      1
True        False   True          True              False     1

Email versus both-phone availability:


Both phones missing,False,True
Email missing,,
False,719,11
True,34,1


Rows with neither phone nor email:


,n_mero_registre,nom_de_l_entitat,naturalesa,cif_entitat,adre_a,codi_postal,municipi,tel_fon,tel_fon_m_bil,correu_electr_nic,web
273,392.0,AssociacióEsperitHumanitarid'Espanyaal'Àfrica-...,Associació,G65245334,"Xifre,75baixos",8026,Barcelona,NaN,NaN,NaN,NaN


Missing rates by CIF availability; descriptive association only:


,rows,tel_fon missing %,tel_fon_m_bil missing %,correu_electr_nic missing %,web missing %
cif_entitat,,,,,
False,682,30.5,26.7,3.8,18.9
True,83,15.7,36.1,10.8,30.1


CIF absence is not identical to absence of any direct-contact field:


Any direct-contact field missing,False,True
CIF missing,,
False,287,395
True,37,46


### Joint missingness — conclusions

- 12 rows lack both phones: 11 still contain email; one lacks email and web as well (registration 392).
- Of the 35 rows without email, 34 contain at least one phone.
- CIF absence does not coincide with incomplete direct-contact fields: 37/83 rows without CIF have all three direct-contact fields, while 395/682 rows with CIF lack at least one.
- Legal-form counts and rates above show descriptive differences. Neither these tables nor overlapping gaps establish why data is absent.

### Ausencias conjuntas — conclusiones

- En 12 filas faltan ambos teléfonos: 11 conservan correo; una carece también de correo y web (registro 392).
- De las 35 filas sin correo, 34 tienen al menos un teléfono.
- La ausencia de CIF no coincide con contactos incompletos: 37/83 filas sin CIF tienen los tres campos de contacto directo; 395/682 con CIF carecen de alguno.
- Los recuentos y porcentajes por naturaleza muestran diferencias descriptivas; no explican la causa de las ausencias.


## Cross-column text issues / Problemas de texto entre columnas

| Field | Cells without whitespace | Literal XML example |
| --- | --- | --- |
| `nom_de_l_entitat` | 764/765; the other contains a newline | `AccióSolidàriadelVallès` |
| `naturalesa` | 765/765 | `AssociacióEstatal` |
| `adre_a` | 765/765 | `PladelVinyet,9BlocDbaixos` |
| `municipi` | 765/765 | `SantCugatdelVallès` |

- These columns contain joined-word examples already in XML. Counts measure whitespace absence, not defective values; single-word labels need no spaces.
- Email domain dots and repeated web schemes are separate observed issues. Absence of spaces alone is not an email/URL defect.
- The earlier source is needed to establish cause or intended text. No common cause is established; no text is repaired.

- Hay palabras unidas en estas columnas, ya en XML. Los recuentos no equivalen a errores: los valores de una palabra no requieren espacios.
- Puntos ausentes en dominios email y esquemas web repetidos son observaciones distintas. No tener espacios no es, por sí solo, un defecto de email/URL.
- La causa y el texto previsto requieren una fuente anterior. No se establece causa común ni se repara texto.


## Comparison with the global schema / Comparación con el esquema global

Candidate correspondences only. No `Entidad`, `Localidad` or `Provincia` instances are created, and no integration rules are applied.

Correspondencias conceptuales candidatas; no se generan entidades ni se ejecutan reglas de integración. `str | None` admite valores nulos, pero los campos no tienen valores por defecto en las dataclasses.

In [41]:
schema_path = Path("../data/global_schema.py")
spec = importlib.util.spec_from_file_location("observed_global_schema", schema_path)
global_schema = importlib.util.module_from_spec(spec)
import sys
sys.modules[spec.name] = global_schema
spec.loader.exec_module(global_schema)
for class_name in ["Entidad", "Localidad", "Provincia"]:
    schema_class = getattr(global_schema, class_name)
    print(class_name)
    display(pd.DataFrame([{"field": field.name, "declared_type": str(field.type)}
                          for field in fields(schema_class)]))
print("Allowed Ambito values:")
display(pd.Series([value.value for value in global_schema.Ambito], name="Ambito"))

Entidad


,field,declared_type
0,cod_entidad,<class 'str'>
1,nombre,<class 'str'>
2,ambito,observed_global_schema.Ambito | None
3,direccion,<class 'str'>
4,codigo_postal,str | None
5,longitud,float | None
6,latitud,float | None
7,descripcion,str | None
8,contacto,str | None
9,url,str | None


Localidad


,field,declared_type
0,codigo,<class 'str'>
1,nombre,<class 'str'>
2,en_provincia,<class 'observed_global_schema.Provincia'>


Provincia


,field,declared_type
0,codigo,<class 'str'>
1,nombre,<class 'str'>


Allowed Ambito values:


0                                      MAYORES
1                                 DISCAPACIDAD
2                                 SALUD_MENTAL
3                          INFANCIA_Y_JUVENTUD
4                                        MUJER
5                                    MIGRACION
6                             INCLUSION_SOCIAL
7                        EDUCACION_Y_FORMACION
8                   EMPLEO_E_INSERCION_LABORAL
9              SALUD_Y_ATENCION_SOCIOSANITARIA
10    VOLUNTARIADO_Y_PARTICIPACION_COMUNITARIA
11            CULTURA_Y_DESARROLLO_COMUNITARIO
Name: Ambito, dtype: str

| Source column | Candidate schema attribute | Observation / Observación |
| --- | --- | --- |
| `n_mero_registre` | `Entidad.cod_entidad` | Complete and unique here; literal identifiers include dots; pandas type differs from schema `str`. |
| `nom_de_l_entitat` | `Entidad.nombre` | Complete; joined words already in XML. |
| `naturalesa` | No dedicated attribute | Legal form does not determine `Ambito`. |
| `cif_entitat` | No dedicated attribute | Missing/repeated; does not provide a complete unique key. |
| `adre_a` | `Entidad.direccion` | Complete composite strings; shared addresses are not unique keys. |
| `codi_postal` | `Entidad.codigo_postal` | Five-digit XML text; numeric inference loses leading zeros. |
| `municipi` | `Localidad.nombre` | Name only; municipality code and province relationship absent. |
| `tel_fon` | `Entidad.contacto` | Contact component; some cells contain several numbers or extensions. |
| `tel_fon_m_bil` | `Entidad.contacto` | Complementary contact component; not necessarily mobile-only. |
| `correu_electr_nic` | `Entidad.contacto` | Domain punctuation and multiple-contact anomalies. |
| `web` | `Entidad.url` | Dotless host strings and repeated schemes require source review. |

### Global schema — conclusions

- All 11 source fields are considered. Coordinates, explicit description and an activity classification establishing `Ambito` are absent.
- The required `Entidad.en_localidad` object cannot be fully populated from explicit fields alone: locality code and province code/name are missing.
- The schema has one optional `contacto` string, with no representation rule for the three contact components or multiple values. No selection, joining or splitting policy is defined here.
- Registration is a candidate key within this file. Repeated CIFs, addresses or contacts do not establish entity identity; cross-source identity remains unresolved.
- Source syntax issues and geographic associations are documented. No reconstruction, correction, record removal or merge is performed.

### Esquema global — conclusiones

- Se consideran los 11 campos. Faltan coordenadas, descripción explícita y clasificación de actividad que determine `Ambito`.
- `Entidad.en_localidad` es obligatorio; no se puede completar solo con campos explícitos: faltan código municipal y código/nombre de provincia.
- `contacto` es un único texto opcional; no define cómo representar tres componentes ni varios contactos por celda. No se fija política de selección, unión o separación.
- Registro es candidato a clave dentro de este archivo. CIF, dirección o contactos compartidos no demuestran identidad; queda abierta la identidad entre fuentes.
- Se documentan anomalías de sintaxis y relaciones geográficas. No se reconstruyen valores, corrigen datos, eliminan filas ni fusionan registros.


In [42]:
pd.testing.assert_frame_equal(df, df_snapshot)
print("Verified: df is unchanged after all diagnostic checks.")

Verified: df is unchanged after all diagnostic checks.
